# 17 · Top 100 LeetCode questions — part 3: linked lists and trees

After this notebook you can solve the 20 most-asked **easy and medium** linked-list and binary-tree questions on
your own — and, more importantly, recognise *which trick* a brand-new one wants. Linked lists come down to a few
moves with fingers (pointers): walk, flip a hand, run fast and slow, add a pretend carriage at the front. Trees come
down to one big idea — **recursion**: *ask your children, then tell your parent* — plus a queue when the question
says "level". Every answer is plain Python 3, checked with `assert`s, and comes with an animation you can step
through node by node.

**Time:** ~60 min · **Runs:** offline on CPU in seconds · **Needs:** [05 · Linked lists](05_linked_lists.ipynb), [09 · Trees and BST](09_trees_and_bst.ipynb) (and [08 · Recursion](08_recursion_and_backtracking.ipynb) if recursion feels new)

## How to use this notebook

1. Read the 🧸 story and the question.
2. **Cover the code and try it yourself** — 10 minutes, even if you only get the idea in words.
3. Run the answer, then press **▶ play** on the animation and watch every step (or step with **next ▶**).
4. Tomorrow, solve it again from a blank cell.

Colours always mean the same thing: **yellow** = looking at it now, **green** = the answer, **red** = no,
**grey** = finished, **blue** = inside the window (here: the slow finger, or a node still waiting for its children),
**orange** = moving (here: the fast finger, or a hand that was just flipped), **purple** = just added.

## The 20 in this notebook

| # | Problem | LeetCode | Level | Trick |
|---|---|---|---|---|
| 41 | Reverse Linked List | [206](https://leetcode.com/problems/reverse-linked-list/) | 🟢 Easy | three fingers: `prev`, `cur`, `nxt` |
| 42 | Merge Two Sorted Lists | [21](https://leetcode.com/problems/merge-two-sorted-lists/) | 🟢 Easy | dummy head + take the smaller front |
| 43 | Linked List Cycle | [141](https://leetcode.com/problems/linked-list-cycle/) | 🟢 Easy | fast and slow (tortoise and hare) |
| 44 | Middle of the Linked List | [876](https://leetcode.com/problems/middle-of-the-linked-list/) | 🟢 Easy | fast and slow |
| 45 | Remove Nth Node From End of List | [19](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) | 🟡 Medium | two fingers `n` apart + dummy |
| 46 | Add Two Numbers | [2](https://leetcode.com/problems/add-two-numbers/) | 🟡 Medium | walk both, carry the 1 |
| 47 | Palindrome Linked List | [234](https://leetcode.com/problems/palindrome-linked-list/) | 🟢 Easy | middle + reverse the back half + compare |
| 48 | Intersection of Two Linked Lists | [160](https://leetcode.com/problems/intersection-of-two-linked-lists/) | 🟢 Easy | at the end, jump to the other train |
| 49 | LRU Cache | [146](https://leetcode.com/problems/lru-cache/) | 🟡 Medium | hash map + doubly linked list (`OrderedDict`) |
| 50 | Maximum Depth of Binary Tree | [104](https://leetcode.com/problems/maximum-depth-of-binary-tree/) | 🟢 Easy | recursion: 1 + max(children) |
| 51 | Invert Binary Tree | [226](https://leetcode.com/problems/invert-binary-tree/) | 🟢 Easy | recursion: swap the children |
| 52 | Same Tree | [100](https://leetcode.com/problems/same-tree/) | 🟢 Easy | recursion on two trees at once |
| 53 | Symmetric Tree | [101](https://leetcode.com/problems/symmetric-tree/) | 🟢 Easy | mirror pairs: outside with outside |
| 54 | Diameter of Binary Tree | [543](https://leetcode.com/problems/diameter-of-binary-tree/) | 🟢 Easy | return height, remember the best |
| 55 | Binary Tree Level Order Traversal | [102](https://leetcode.com/problems/binary-tree-level-order-traversal/) | 🟡 Medium | BFS with a queue, one level at a time |
| 56 | Binary Tree Right Side View | [199](https://leetcode.com/problems/binary-tree-right-side-view/) | 🟡 Medium | BFS, keep the last of each level |
| 57 | Validate Binary Search Tree | [98](https://leetcode.com/problems/validate-binary-search-tree/) | 🟡 Medium | pass the allowed range `(low, high)` down |
| 58 | Kth Smallest Element in a BST | [230](https://leetcode.com/problems/kth-smallest-element-in-a-bst/) | 🟡 Medium | in-order is sorted; stop at `k` |
| 59 | Lowest Common Ancestor of a Binary Tree | [236](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/) | 🟡 Medium | "did you find `p` or `q`?" from both sides |
| 60 | Construct Binary Tree from Preorder and Inorder Traversal | [105](https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/) | 🟡 Medium | preorder names the root, inorder splits left/right |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "03_dsa", *Path.cwd().parents]:
    if (_p / "dsa_anim.py").exists():
        sys.path.insert(0, str(_p)); break
from dsa_anim import Movie, arr, bars, hmap, stack, queue, chain, chain_of, tree, grid, graph, note, say

**Shared helpers.** LeetCode hands you linked lists and trees as node objects, but writes the examples as Python
lists. These helpers turn one into the other, so every `assert` below can use the same lists LeetCode shows.
Trees use LeetCode's *level-order* format: `[3, 9, 20, None, None, 15, 7]` means root `3`, children `9` and `20`,
`9` has no children, `20` has children `15` and `7`.

In [ ]:
import random
from collections import deque, OrderedDict


class ListNode:
    "One carriage of a linked-list train: a value, and a hand holding the next carriage (or None)."
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next


def build_list(values):
    "Turn [1, 2, 3] into the train 1 → 2 → 3 and return its head (None for [])."
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head


def to_list(head):
    "Walk the train from the head and collect the values into a Python list."
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


class TreeNode:
    "One node of a binary tree: a value and up to two children."
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def build_tree(values):
    "Build a tree from LeetCode's level-order list, e.g. [3, 9, 20, None, None, 15, 7]; returns the root."
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def to_level_list(root):
    "The opposite of build_tree: level-order values with None for gaps, trailing Nones trimmed."
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def nodes_of(head):
    "The carriages of a train as a Python list of nodes (not values), front first."
    out = []
    while head:
        out.append(head)
        head = head.next
    return out


def train(nodes, title="", hi=None, ptr=None):
    "Draw the carriages `nodes` in this fixed order with their REAL .next hands, so a flipped hand points left."
    pos = {id(nd): i for i, nd in enumerate(nodes)}
    nexts = [pos.get(id(nd.next)) if nd.next is not None else None for nd in nodes]
    return chain([nd.val for nd in nodes], title, nexts,
                 {pos[id(k)]: r for k, r in (hi or {}).items()},
                 {name: pos[id(nd)] for name, nd in (ptr or {}).items() if nd is not None})


assert to_list(build_list([1, 2, 3])) == [1, 2, 3] and to_list(build_list([])) == []
assert to_level_list(build_tree([3, 9, 20, None, None, 15, 7])) == [3, 9, 20, None, None, 15, 7]
assert to_level_list(build_tree([1, None, 2, 3])) == [1, None, 2, 3] and build_tree([]) is None
print("helpers ready")

## The tricks in this notebook, in pictures

**1 · A linked list is a train.** Each carriage holds the hand of the next one. You only know the front carriage
(the `head`), and hands point **one way**, so the only move is "walk forward". To take a carriage out, you stand on
the carriage **before** it and re-hook its hand. If the carriage to remove might be the very first one, put a
pretend carriage `D` (a **dummy**) in front, so there is *always* a carriage before.

In [ ]:
def walk_and_rehook_movie(values, remove):
    head = build_list(values)
    m = Movie("A linked list is a train", "Walk from the front with a finger; take a carriage out by re-hooking one hand")
    m.step("A linked list is a **train**: each carriage holds the hand of the next. We only know the front one, the `head`.",
           chain_of(head, ptr={"head": head}))
    cur, count, before, seen = head, 0, None, []
    while cur:
        count += 1
        m.step(f"Our finger `cur` is on carriage `{cur.val}`. To move on we follow its hand: `cur = cur.next`.",
               chain_of(head, hi={**{s: "done" for s in seen}, cur: "look"}, ptr={"cur": cur}), note(counted=count))
        if cur.next is not None and cur.next.val == remove:
            before = cur                        # the carriage just before the one we want out
        seen.append(cur)
        cur = cur.next
    m.step(f"`cur` fell off the end (`None`): {count} carriages. Hands only point **forward**, so we can never walk back.",
           chain_of(head, hi={s: "done" for s in seen}), note(counted=count))
    gone = before.next
    m.step(f"To take `{remove}` out, stand on the carriage **before** it, `{before.val}`.",
           chain_of(head, hi={before: "look", gone: "bad"}, ptr={"before": before}))
    before.next = gone.next                     # re-hook one hand: skip over the carriage
    m.step(f"`{before.val}` lets go of `{remove}` and holds `{before.next.val}` instead. Nobody holds `{remove}` now, so it has left the train ✅",
           chain_of(head, hi={before: "good", before.next: "good"}, ptr={"before": before}))
    return to_list(head), m

answer, m = walk_and_rehook_movie([7, 3, 9, 4, 6], 9)
assert answer == [7, 3, 4, 6]
m.show()

**2 · Fast and slow fingers (the tortoise 🐢 and the hare 🐇).** Two fingers start at the front. The tortoise steps
one carriage at a time, the hare jumps two. When the hare reaches the end, the tortoise is exactly in the
**middle**. If the track is a **loop**, the hare laps the tortoise and they meet. Use it for: middle, cycle,
"n-th from the end" (start them apart instead), palindrome.

In [ ]:
def fast_slow_movie(values):
    head = build_list(values)
    m = Movie("Fast and slow fingers", "The hare 🐇 jumps two carriages, the tortoise 🐢 walks one")
    slow = fast = head
    m.step("🐢 `slow` and 🐇 `fast` both start on the front carriage.", chain_of(head, hi={slow: "look"}, ptr={"slow": slow, "fast": fast}))
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        m.step(f"🐢 walks 1 to `{slow.val}`. 🐇 jumps 2 to `{fast.val if fast else 'off the end'}`. The hare has gone twice as far.",
               chain_of(head, hi={slow: "win", **({fast: "swap"} if fast else {})}, ptr={"slow": slow, "fast": fast}))
    m.step(f"🐇 cannot jump two more, so it is at the end. 🐢 went half as far, so it stands on the **middle**: `{slow.val}` ✅",
           chain_of(head, hi={slow: "good"}, ptr={"slow": slow}))
    return slow.val, m

answer, m = fast_slow_movie([1, 2, 3, 4, 5, 6, 7, 8, 9])
assert answer == 5
m.show()

**3 · Recursion: ask your children, then tell your parent.** Almost every tree question is answered like this:
a node asks its left child and its right child the *same* question, waits for their answers, mixes them with its
own value, and hands one answer up to its parent. An empty spot (`None`) answers straight away — that is the
**base case**, the place where the asking stops. The waiting nodes form a pile (the *call stack*). Below, every
node answers *"how many nodes are in your part of the tree?"* with `1 + left answer + right answer`.

In [ ]:
def count_nodes_movie(values):
    root = build_tree(values)
    m = Movie("Ask your children, then tell your parent", "How many nodes does this tree have? Recursion, one call at a time")
    waiting, told = [], {}                      # the pile of nodes waiting for answers, and every answer given
    def pic(extra):
        hi = {**{nd: "done" for nd in told_nodes}, **{nd: "win" for nd in waiting}, **extra}
        return (tree(root, "the tree", hi=hi), stack([nd.val for nd in waiting], "waiting (call stack)"),
                hmap(told, "node → its answer"))
    told_nodes = []
    m.step("Each node asks its two children *how many are below you?*, adds 1 for itself, and tells its parent. Empty spots answer `0`.",
           *pic({}))
    def count(node):
        if node is None:                        # base case: nobody here
            return 0
        waiting.append(node)
        m.step(f"Node `{node.val}` must ask its children first, so it **waits** on the pile.", *pic({node: "look"}))
        left = count(node.left)                 # ask the left child ...
        right = count(node.right)               # ... and the right child
        waiting.pop()
        total = 1 + left + right                # mix the answers with myself
        told[node.val] = total
        told_nodes.append(node)
        m.step(f"Node `{node.val}` heard left = `{left}`, right = `{right}`. It tells its parent 1 + {left} + {right} = **{total}**.",
               *pic({node: "new"}))
        return total
    total = count(root)
    m.step(f"The root answered last: the tree has **{total}** nodes ✅ Each node was visited once.", *pic({root: "good"}))
    return total, m

answer, m = count_nodes_movie([1, 2, 3, 4, 5])
assert answer == 5
m.show()

---

## 41 · Reverse Linked List · [LeetCode 206](https://leetcode.com/problems/reverse-linked-list/) · 🟢 Easy

**🧸 Like you're five:** A train of carriages, each holding the hand of the carriage in front. Turn every hand around so the
        train points the other way. Before a carriage lets go, a friend must hold on to the rest of the train, or it
        rolls away!

**The question:** Given the `head` of a singly linked list, reverse the list and return the new head. The list has 0 to 5000 nodes.

| Input | Output |
|---|---|
| `head = [1, 2, 3, 4, 5]` | `[5, 4, 3, 2, 1]` |
| `head = [1, 2]` | `[2, 1]` |
| `head = []` | `[]` |

**🔎 How to spot it:** "**reverse** a linked list" — or any question that needs part of a list backwards (palindrome
        list, reverse in groups, reorder list). Reversing is a building block: learn it until you can type it blind.

**💡 The trick (three fingers: prev, cur, nxt):** Keep three fingers. `prev` = the part already turned around (starts as `None`), `cur` = the carriage
        we are turning, `nxt` = the rest of the train. For each carriage: **(1)** `nxt = cur.next` (save the rest),
        **(2)** `cur.next = prev` (flip the hand), **(3)** step forward: `prev, cur = cur, nxt`. When `cur` is `None`,
        `prev` is the new head. The classic bug: flipping the hand *before* saving `nxt`, which loses the rest of the
        list.

**⏱ Cost:** **O(n)** time (each node once) · **O(1)** extra space. The recursive version is also O(n) time but uses O(n) stack space; copying the values into a Python list and rebuilding uses O(n) space.

In [ ]:
def reverse_list(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next          # 1. remember the rest of the train
        cur.next = prev         # 2. flip this carriage's hand backwards
        prev, cur = cur, nxt    # 3. step forward
    return prev                 # prev is the old tail = the new head

assert to_list(reverse_list(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert to_list(reverse_list(build_list([1, 2]))) == [2, 1]
assert to_list(reverse_list(build_list([7]))) == [7]
assert reverse_list(None) is None
rng = random.Random(0)
for _ in range(300):
    vals = [rng.randint(-9, 9) for _ in range(rng.randint(0, 8))]
    assert to_list(reverse_list(build_list(vals))) == vals[::-1]
print(to_list(reverse_list(build_list([1, 2, 3, 4, 5]))))

In [ ]:
def reverse_list_movie(values):
    head = build_list(values)
    nodes, cur = [], head
    while cur:                                  # keep the carriages in their old places for the picture
        nodes.append(cur); cur = cur.next
    m = Movie("Reverse Linked List", "Turn every hand around, one carriage at a time")
    prev, cur, flipped = None, head, []
    m.step(f"Every hand points right. We want them all to point **left**, so the train reads {values[::-1]}. `prev` starts as nothing.",
           train(nodes, hi={cur: "look"}, ptr={"cur": cur}), note(prev=None))
    while cur:
        nxt = cur.next
        what = f"carriage `{nxt.val}`" if nxt else "nothing — this is the last carriage"
        m.step(f"Before `{cur.val}` lets go, `nxt` holds on to {what}, so the rest of the train cannot roll away.",
               train(nodes, hi={**{f: "done" for f in flipped}, cur: "look"}, ptr={"prev": prev, "cur": cur, "nxt": nxt}))
        cur.next = prev
        flipped.append(cur)
        to = f"`{prev.val}`" if prev else "nothing (∅)"
        m.step(f"Flip! `{cur.val}` now holds {to}. Then `prev` steps onto `{cur.val}` and `cur` onto `nxt`.",
               train(nodes, hi={**{f: "done" for f in flipped}, cur: "swap"}, ptr={"prev": cur, "cur": nxt}))
        prev, cur = cur, nxt
    m.step(f"`cur` fell off the end. `prev` = `{prev.val}` is the new front: **{to_list(prev)}** ✅",
           train(nodes, hi={nd: "good" for nd in nodes}, ptr={"head": prev}))
    return to_list(prev), m

values = [1, 2, 3, 4, 5]
answer, m = reverse_list_movie(values)
assert answer == to_list(reverse_list(build_list(values))) == [5, 4, 3, 2, 1]
m.show()

---

## 42 · Merge Two Sorted Lists · [LeetCode 21](https://leetcode.com/problems/merge-two-sorted-lists/) · 🟢 Easy

**🧸 Like you're five:** Two lines of children, each already sorted from shortest to tallest. To make one sorted line, keep
        looking at the two children at the front and send the shorter one to the new line. When one line is empty,
        the whole other line just walks over.

**The question:** Merge two sorted linked lists `list1` and `list2` into one sorted list made by splicing together their nodes, and return its head. Either list may be empty.

| Input | Output |
|---|---|
| `list1 = [1, 2, 4], list2 = [1, 3, 4]` | `[1, 1, 2, 3, 4, 4]` |
| `list1 = [], list2 = []` | `[]` |
| `list1 = [], list2 = [0]` | `[0]` |

**🔎 How to spot it:** "**merge** two **sorted**" lists → compare the two fronts. Building a new list whose first node is
        not known yet → start from a **dummy** node, so you never need an `if head is None` special case.

**💡 The trick (dummy head + take the smaller front):** Make a dummy node and a finger `tail` on it. While both lists have nodes, hook the smaller front onto
        `tail.next`, move that list forward, and move `tail` forward. When one list runs out, hook the *whole* other
        list on in one go (`tail.next = list1 or list2`) — it is already sorted. Return `dummy.next`, not `dummy`.

**⏱ Cost:** **O(n + m)** time (each node is hooked once) · **O(1)** extra space (we re-use the nodes). Brute force: pour all values into a Python list, sort, rebuild — O((n + m) log(n + m)) time and O(n + m) space.

In [ ]:
def merge_two_lists(list1, list2):
    dummy = tail = ListNode()                  # pretend first carriage; tail = end of the new train
    while list1 and list2:
        if list1.val <= list2.val:             # the smaller front goes next
            tail.next, list1 = list1, list1.next
        else:
            tail.next, list2 = list2, list2.next
        tail = tail.next
    tail.next = list1 or list2                 # one list is empty: hook on the rest of the other
    return dummy.next

assert to_list(merge_two_lists(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]
assert to_list(merge_two_lists(build_list([]), build_list([]))) == []
assert to_list(merge_two_lists(build_list([]), build_list([0]))) == [0]
assert to_list(merge_two_lists(build_list([-3, 5]), build_list([-7]))) == [-7, -3, 5]
rng = random.Random(0)
for _ in range(300):
    a = sorted(rng.randint(-9, 9) for _ in range(rng.randint(0, 6)))
    b = sorted(rng.randint(-9, 9) for _ in range(rng.randint(0, 6)))
    assert to_list(merge_two_lists(build_list(a), build_list(b))) == sorted(a + b)
print(to_list(merge_two_lists(build_list([1, 2, 4]), build_list([1, 3, 4]))))

In [ ]:
def merge_two_lists_movie(a, b):
    list1, list2 = build_list(a), build_list(b)
    dummy = tail = ListNode("D")
    m = Movie("Merge Two Sorted Lists", "Zip two sorted trains into one sorted train")
    def merged():                               # the new train from dummy.next up to tail
        out, cur = [], dummy.next
        while cur is not None:
            out.append(cur.val)
            if cur is tail:
                break
            cur = cur.next
        return ["D"] + out
    def pic(extra=None):
        k = len(merged()) - 1
        return (chain_of(list1, "list1 (what is left)", hi={list1: "look"}, ptr={"list1": list1}),
                chain_of(list2, "list2 (what is left)", hi={list2: "look"}, ptr={"list2": list2}),
                chain(merged(), "merged so far (D = dummy)", hi=extra or {k: "new"}, ptr={"tail": k}))
    m.step("Two sorted trains. The new train starts with a pretend carriage `D` (the **dummy**); `tail` marks its end.", *pic({}))
    while list1 and list2:
        if list1.val <= list2.val:
            x, side = list1.val, "list1"
            tail.next, list1 = list1, list1.next
        else:
            x, side = list2.val, "list2"
            tail.next, list2 = list2, list2.next
        tail = tail.next
        other = list2 if side == "list1" else list1
        m.step(f"Fronts `{x}` and `{other.val if other else '-'}`: `{x}` is smaller (a tie takes list1), so it hooks onto the new train and {side} moves on.",
               *pic())
    rest = "list1" if list1 else "list2"
    tail.next = list1 or list2
    m.step(f"One train is empty. The rest of {rest} is already sorted, so hook it all on in **one** move.",
           chain_of(list1, "list1 (what is left)"), chain_of(list2, "list2 (what is left)"),
           chain_of(dummy, "merged (D = dummy)", hi={tail.next: "new"} if tail.next else {}))
    answer = to_list(dummy.next)
    m.step(f"Return `dummy.next` (skip the pretend carriage): **{answer}** ✅",
           chain_of(dummy.next, "answer", hi={nd: "good" for nd in nodes_of(dummy.next)}))
    return answer, m

a, b = [1, 2, 4], [1, 3, 4]
answer, m = merge_two_lists_movie(a, b)
assert answer == to_list(merge_two_lists(build_list(a), build_list(b))) == [1, 1, 2, 3, 4, 4]
m.show()

---

## 43 · Linked List Cycle · [LeetCode 141](https://leetcode.com/problems/linked-list-cycle/) · 🟢 Easy

**🧸 Like you're five:** A tortoise and a hare run along a train track. If the track just ends, the fast hare reaches the end
        first. But if the track goes round in a circle, the hare runs round and round and must, one day, catch up with
        the tortoise from behind.

**The question:** Given `head`, return `True` if the linked list has a cycle (some node can be reached again by following `next`), otherwise `False`. Follow-up: use **O(1)** extra memory.

| Input | Output |
|---|---|
| `head = [3, 2, 0, -4], tail joins index 1` | `True` |
| `head = [1, 2], tail joins index 0` | `True` |
| `head = [1], no cycle` | `False` |

**🔎 How to spot it:** "does the list have a **cycle / loop**?", especially with "**O(1) memory**" → fast and slow
        pointers. (With memory allowed, a `set` of visited nodes works too.)

**💡 The trick (fast and slow (tortoise and hare)):** `slow` moves one node, `fast` moves two. If `fast` reaches `None`, the list ends — no cycle. If there
        is a loop, both end up inside it, and each step the gap between them shrinks by exactly one, so `fast` lands
        **on** `slow` rather than jumping over it. Compare nodes with `is` (same carriage), not values — two carriages
        may carry the same number. Check `fast and fast.next` before jumping, or you crash on `None.next`.

**⏱ Cost:** **O(n)** time · **O(1)** space. Remembering every visited node in a set is also O(n) time but O(n) space.

In [ ]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:          # the hare can still jump two
        slow = slow.next               # 🐢 one step
        fast = fast.next.next          # 🐇 two steps
        if slow is fast:               # same carriage → the hare lapped the tortoise
            return True
    return False                       # the hare fell off the end: no loop

def build_cycle(values, pos):
    "A train whose last carriage holds carriage number `pos` (pos = -1: no loop)."
    head = build_list(values)
    nodes, cur = [], head
    while cur:
        nodes.append(cur); cur = cur.next
    if pos >= 0:
        nodes[-1].next = nodes[pos]
    return head

assert has_cycle(build_cycle([3, 2, 0, -4], 1)) is True
assert has_cycle(build_cycle([1, 2], 0)) is True
assert has_cycle(build_cycle([1], -1)) is False
assert has_cycle(build_cycle([1], 0)) is True           # a carriage holding its own hand
assert has_cycle(None) is False
rng = random.Random(0)
for _ in range(300):
    n = rng.randint(1, 9)
    pos = rng.randint(-1, n - 1)
    assert has_cycle(build_cycle([rng.randint(0, 3) for _ in range(n)], pos)) is (pos >= 0)
print(has_cycle(build_cycle([3, 2, 0, -4], 1)))

In [ ]:
def has_cycle_movie(values, pos):
    head = build_cycle(values, pos)
    m = Movie("Linked List Cycle", "Does the track go round in a circle? 🐢 walks 1, 🐇 jumps 2")
    slow = fast = head
    m.step(f"The last carriage `{values[-1]}` holds `{values[pos]}` again (the red curve), making a loop. Both runners start at the front.",
           chain_of(head, hi={slow: "look"}, ptr={"slow 🐢": slow, "fast 🐇": fast}))
    steps = 0
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        steps += 1
        if slow is fast:
            m.step(f"🐇 landed on the same carriage as 🐢 (`{slow.val}`)! Only a loop lets the fast one come up from behind → **True** ✅",
                   chain_of(head, hi={slow: "good"}, ptr={"slow 🐢": slow, "fast 🐇": fast}), note(steps=steps))
            return True, m
        m.step(f"🐢 walks to `{slow.val}`, 🐇 jumps to `{fast.val if fast else '∅'}`. Not the same carriage yet.",
               chain_of(head, hi={slow: "win", **({fast: "swap"} if fast else {})}, ptr={"slow 🐢": slow, "fast 🐇": fast}),
               note(steps=steps))
    m.step("🐇 fell off the end, so the track has an end → **False**.", chain_of(head))
    return False, m

values, pos = [1, 2, 3, 4, 5, 6, 7], 2
answer, m = has_cycle_movie(values, pos)
assert answer == has_cycle(build_cycle(values, pos)) is True
m.show()

---

## 44 · Middle of the Linked List · [LeetCode 876](https://leetcode.com/problems/middle-of-the-linked-list/) · 🟢 Easy

**🧸 Like you're five:** You want to stand in the middle of a train, but you cannot see how long it is. Send a fast friend who
        takes two steps every time you take one. When your friend reaches the end, you are in the middle.

**The question:** Given the `head` of a singly linked list, return the middle node. If there are two middle nodes (even length), return the **second** one.

| Input | Output |
|---|---|
| `head = [1, 2, 3, 4, 5]` | `[3, 4, 5]` |
| `head = [1, 2, 3, 4, 5, 6]` | `[4, 5, 6]` |
| `head = [1]` | `[1]` |

**🔎 How to spot it:** "the **middle** of a linked list" (also the first step of *palindrome list*, *reorder list*, *sort
        list*) → fast and slow in one pass. You do not need to count the length first.

**💡 The trick (fast and slow):** Start both at `head`. While `fast and fast.next`: `slow` one step, `fast` two steps. Then `slow` is the
        middle. This loop condition gives the **second** middle for an even length. (If a question wants the *first*
        middle, loop while `fast.next and fast.next.next` instead — the off-by-one people get wrong.)

**⏱ Cost:** **O(n)** time · **O(1)** space. Counting first and then walking `n // 2` steps is also O(n), just two passes.

In [ ]:
def middle_node(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next           # one step
        fast = fast.next.next      # two steps
    return slow                    # fast is at (or past) the end, slow is half-way

assert to_list(middle_node(build_list([1, 2, 3, 4, 5]))) == [3, 4, 5]
assert to_list(middle_node(build_list([1, 2, 3, 4, 5, 6]))) == [4, 5, 6]   # second middle
assert to_list(middle_node(build_list([1]))) == [1]
assert to_list(middle_node(build_list([1, 2]))) == [2]
for n in range(1, 30):
    vals = list(range(n))
    assert middle_node(build_list(vals)).val == n // 2
print(to_list(middle_node(build_list([1, 2, 3, 4, 5, 6]))))

In [ ]:
def middle_node_movie(values):
    head = build_list(values)
    m = Movie("Middle of the Linked List", "🐇 jumps two, 🐢 walks one; when 🐇 is done, 🐢 is in the middle")
    slow = fast = head
    m.step(f"{len(values)} carriages (an even number, so there are two middles; we want the **second**). Both start at the front.",
           chain_of(head, hi={slow: "look"}, ptr={"slow": slow, "fast": fast}))
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        where = f"`{fast.val}`" if fast else "off the end (∅)"
        m.step(f"🐢 `slow` walks to `{slow.val}`, 🐇 `fast` jumps to {where}.",
               chain_of(head, hi={slow: "win", **({fast: "swap"} if fast else {})}, ptr={"slow": slow, "fast": fast}))
    m.step(f"🐇 cannot jump two any more, so we stop. 🐢 stands on `{slow.val}`: answer **{to_list(slow)}** ✅",
           chain_of(head, hi={nd: "good" for nd in nodes_of(slow)}, ptr={"slow": slow}))
    return to_list(slow), m

values = [1, 2, 3, 4, 5, 6, 7, 8]
answer, m = middle_node_movie(values)
assert answer == to_list(middle_node(build_list(values))) == [5, 6, 7, 8]
m.show()

---

## 45 · Remove Nth Node From End of List · [LeetCode 19](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) · 🟡 Medium

**🧸 Like you're five:** Take out the 2nd carriage from the back, but you can only walk from the front. Hold a stick that is
        exactly 2 carriages long: when the front end of the stick falls off the train, the back end is right before
        the carriage to remove.

**The question:** Given the `head` of a linked list, remove the `n`-th node from the **end** and return the head. `1 ≤ n ≤ length`. Follow-up: one pass.

| Input | Output |
|---|---|
| `head = [1, 2, 3, 4, 5], n = 2` | `[1, 2, 3, 5]` |
| `head = [1], n = 1` | `[]` |
| `head = [1, 2], n = 1` | `[1]` |

**🔎 How to spot it:** "**n-th from the end**" of a linked list (you cannot index from the back) → two pointers with a
        fixed **gap**. "Remove" a node that might be the head → **dummy** node in front.

**💡 The trick (two fingers n apart + dummy):** Put a dummy in front. Start `fast` and `slow` on the dummy. Move `fast` ahead `n + 1` steps, so there
        are `n` carriages between them. Now move both together until `fast` falls off the end. `slow` is now the
        carriage **before** the one to remove, so `slow.next = slow.next.next`. The dummy is what makes "remove the
        head" (`[1], n = 1`) work with no special case.

**⏱ Cost:** **O(L)** time, one pass · **O(1)** space. Two-pass version: count the length L, then walk `L - n` steps — also O(L), fine to say first.

In [ ]:
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)              # a carriage before the head, so the head can be removed too
    slow = fast = dummy
    for _ in range(n + 1):                 # make a gap of n carriages between slow and fast
        fast = fast.next
    while fast:                            # walk together until fast falls off the end
        slow, fast = slow.next, fast.next
    slow.next = slow.next.next             # slow is just before the target: skip over it
    return dummy.next

assert to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(remove_nth_from_end(build_list([1]), 1)) == []
assert to_list(remove_nth_from_end(build_list([1, 2]), 1)) == [1]
assert to_list(remove_nth_from_end(build_list([1, 2]), 2)) == [2]         # removing the head
rng = random.Random(0)
for _ in range(300):
    vals = [rng.randint(0, 9) for _ in range(rng.randint(1, 8))]
    n = rng.randint(1, len(vals))
    assert to_list(remove_nth_from_end(build_list(vals), n)) == vals[:len(vals) - n] + vals[len(vals) - n + 1:]
print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))

In [ ]:
def remove_nth_movie(values, n):
    dummy = ListNode("D", build_list(values))
    m = Movie("Remove Nth Node From End of List", f"Take out the carriage number {n} from the back")
    slow = fast = dummy
    m.step(f"Put a pretend carriage `D` (**dummy**) in front. Both fingers start on it. We want to remove #{n} from the end.",
           chain_of(dummy, hi={dummy: "look"}, ptr={"slow": slow, "fast": fast}), note(n=n))
    for k in range(1, n + 2):
        fast = fast.next
        m.step(f"`fast` goes ahead alone: step {k} of {n + 1}. Afterwards `{n}` carriages will sit between the fingers.",
               chain_of(dummy, hi={slow: "win", **({fast: "swap"} if fast else {})}, ptr={"slow": slow, "fast": fast}), note(n=n))
    while fast:
        slow, fast = slow.next, fast.next
        where = f"`{fast.val}`" if fast else "off the end (∅)"
        m.step(f"Both walk one step together (the gap stays the same): `slow` on `{slow.val}`, `fast` {where}.",
               chain_of(dummy, hi={slow: "win", **({fast: "swap"} if fast else {})}, ptr={"slow": slow, "fast": fast}), note(n=n))
    gone = slow.next
    m.step(f"`fast` fell off, so `slow` (`{slow.val}`) is right **before** the target `{gone.val}`.",
           chain_of(dummy, hi={slow: "look", gone: "bad"}, ptr={"slow": slow}), note(n=n))
    slow.next = slow.next.next
    answer = to_list(dummy.next)
    m.step(f"`slow.next = slow.next.next`: `{slow.val}` skips over `{gone.val}`. Return `dummy.next`: **{answer}** ✅",
           chain_of(dummy.next, "answer", hi={nd: "good" for nd in nodes_of(dummy.next)}))
    return answer, m

values, n = [1, 2, 3, 4, 5, 6], 2
answer, m = remove_nth_movie(values, n)
assert answer == to_list(remove_nth_from_end(build_list(values), n)) == [1, 2, 3, 4, 6]
m.show()

---

## 46 · Add Two Numbers · [LeetCode 2](https://leetcode.com/problems/add-two-numbers/) · 🟡 Medium

**🧸 Like you're five:** Adding numbers the way you learned at school: ones first, then tens, then hundreds, and when a column
        makes 10 or more you carry a 1 to the next column. Here the digits are already written ones-first, so you just
        walk both trains from the front.

**The question:** Two non-negative numbers are stored as linked lists with their digits in **reverse order** (ones digit first), one digit per node, no leading zeros except the number 0. Return their sum as a linked list in the same format.

| Input | Output |
|---|---|
| `l1 = [2, 4, 3], l2 = [5, 6, 4]` | `[7, 0, 8]   (342 + 465 = 807)` |
| `l1 = [0], l2 = [0]` | `[0]` |
| `l1 = [9, 9, 9, 9, 9, 9, 9], l2 = [9, 9, 9, 9]` | `[8, 9, 9, 9, 0, 0, 0, 1]` |

**🔎 How to spot it:** "digits stored in **reverse order**" in a linked list → add column by column with a `carry`, exactly
        like on paper. "Build a new list" → dummy head.

**💡 The trick (walk both, carry the 1):** Walk both lists at once with a `carry` that starts at 0. Each round: `total = carry + digit1 + digit2`
        (a missing digit counts as 0), the new node gets `total % 10`, and `carry = total // 10`. Keep going **while
        either list has nodes OR carry is not 0** — forgetting the last carry (`99 + 1 = 100`) is the classic bug, and
        so is stopping when the shorter list ends. Do not convert to a Python `int`: in other languages the numbers
        overflow, and interviewers want the digit walk.

**⏱ Cost:** **O(max(n, m))** time · **O(max(n, m))** space for the answer list (O(1) extra besides it).

In [ ]:
def add_two_numbers(l1, l2):
    dummy = tail = ListNode()
    carry = 0
    while l1 or l2 or carry:                               # keep going while anything is left, even just a carry
        total = carry + (l1.val if l1 else 0) + (l2.val if l2 else 0)
        carry, digit = divmod(total, 10)                   # 14 -> carry 1, digit 4
        tail.next = ListNode(digit)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    return dummy.next

def digits(n):
    "The number n as a ones-first list of digits: 342 -> [2, 4, 3]."
    return [int(c) for c in str(n)[::-1]]

assert to_list(add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_two_numbers(build_list([0]), build_list([0]))) == [0]
assert to_list(add_two_numbers(build_list([9] * 7), build_list([9] * 4))) == [8, 9, 9, 9, 0, 0, 0, 1]
assert to_list(add_two_numbers(build_list([5]), build_list([5]))) == [0, 1]       # the last carry makes a new node
rng = random.Random(0)
for _ in range(300):
    a, b = rng.randint(0, 10**6), rng.randint(0, 10**4)
    assert to_list(add_two_numbers(build_list(digits(a)), build_list(digits(b)))) == digits(a + b)
print(to_list(add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))))

In [ ]:
def add_two_numbers_movie(a, b):
    h1, h2 = build_list(a), build_list(b)
    n1, n2 = int("".join(map(str, a[::-1]))), int("".join(map(str, b[::-1])))
    m = Movie("Add Two Numbers", f"{n1} + {n2}, one column at a time (the ones digit comes first)")
    dummy = tail = ListNode()
    l1, l2, carry = h1, h2, 0
    m.step(f"The trains hold {n1} and {n2} **backwards** (ones first). Add column by column like at school, starting with `carry = 0`.",
           chain_of(h1, "l1", ptr={"l1": l1}), chain_of(h2, "l2", ptr={"l2": l2}), chain_of(None, "answer so far"), note(carry=0))
    while l1 or l2 or carry:
        d1, d2 = (l1.val if l1 else 0), (l2.val if l2 else 0)
        total = carry + d1 + d2
        old = carry
        carry, digit = divmod(total, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        if total >= 10:
            why = f"That is 10 or more: write `{digit}` and **carry 1** to the next column."
        elif not l1 and not l2:
            why = "Both trains are empty, but a carry was left over, so it becomes one more digit!"
        else:
            why = f"Less than 10: write `{digit}`, nothing to carry."
        m.step(f"{d1} + {d2} + carry {old} = **{total}**. {why}",
               chain_of(h1, "l1", hi={l1: "look"} if l1 else {}, ptr={"l1": l1}),
               chain_of(h2, "l2", hi={l2: "look"} if l2 else {}, ptr={"l2": l2}),
               chain_of(dummy.next, "answer so far", hi={tail: "new"}), note(total=total, carry=carry))
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    answer = to_list(dummy.next)
    m.step(f"Nothing left and no carry. Answer **{answer}**, which is {n1 + n2} backwards ({n1} + {n2} = {n1 + n2}) ✅",
           chain_of(h1, "l1"), chain_of(h2, "l2"),
           chain_of(dummy.next, "answer", hi={nd: "good" for nd in nodes_of(dummy.next)}), note(carry=carry))
    return answer, m

a, b = [2, 4, 9], [5, 6, 4, 9]
answer, m = add_two_numbers_movie(a, b)
assert answer == to_list(add_two_numbers(build_list(a), build_list(b))) == digits(942 + 9465)
m.show()

---

## 47 · Palindrome Linked List · [LeetCode 234](https://leetcode.com/problems/palindrome-linked-list/) · 🟢 Easy

**🧸 Like you're five:** A train reads the same from both ends, like `1 2 3 2 1`? You cannot walk backwards on a train. So find
        the middle, turn the back half around, and then walk in from both ends at once, comparing carriages.

**The question:** Given the `head` of a singly linked list, return `True` if it is a palindrome (reads the same forwards and backwards). Follow-up: **O(n)** time and **O(1)** extra space.

| Input | Output |
|---|---|
| `head = [1, 2, 2, 1]` | `True` |
| `head = [1, 2]` | `False` |
| `head = [1]` | `True` |

**🔎 How to spot it:** "**palindrome**" + "**linked list**" + "O(1) space" → you need to read the back half backwards,
        so combine two tricks you already have: fast/slow to find the **middle** (problem 44) and **reverse** (problem
        41).

**💡 The trick (middle + reverse the back half + compare):** **(1)** Fast and slow: when `fast` stops, `slow` is at the middle. **(2)** Reverse the list from `slow`
        to the end. **(3)** Walk one finger from `head` and one from the new back-half head, comparing values, until the
        back-half finger runs out. Any mismatch → `False`. For an odd length the middle carriage is compared with
        itself, which is fine. The easy O(n)-space answer is `vals == vals[::-1]` — say it first. Note that this
        changes the input list; in an interview mention you can reverse the back half again to put it back.

**⏱ Cost:** **O(n)** time · **O(1)** extra space. Copying the values into a Python list is O(n) time and O(n) space.

In [ ]:
def is_palindrome(head):
    slow = fast = head
    while fast and fast.next:            # 1. find the middle
        slow, fast = slow.next, fast.next.next
    prev = None
    while slow:                          # 2. reverse the back half (problem 41)
        nxt = slow.next
        slow.next = prev
        prev, slow = slow, nxt
    left, right = head, prev             # 3. walk in from both ends
    while right:
        if left.val != right.val:
            return False
        left, right = left.next, right.next
    return True

assert is_palindrome(build_list([1, 2, 2, 1])) is True
assert is_palindrome(build_list([1, 2])) is False
assert is_palindrome(build_list([1])) is True
assert is_palindrome(build_list([1, 2, 3, 2, 1])) is True
assert is_palindrome(build_list([1, 2, 3, 1])) is False
rng = random.Random(0)
for _ in range(300):
    half = [rng.randint(0, 2) for _ in range(rng.randint(0, 4))]
    vals = half + [rng.randint(0, 2)] * rng.randint(0, 1) + half[::-1]
    if rng.random() < 0.5 and vals:
        vals[rng.randrange(len(vals))] = rng.randint(0, 2)    # maybe spoil it
    if vals:
        assert is_palindrome(build_list(vals)) == (vals == vals[::-1])
print(is_palindrome(build_list([1, 2, 2, 1])))

In [ ]:
def is_palindrome_movie(values):
    head = build_list(values)
    nodes = nodes_of(head)                    # remember the carriages' places for the picture
    m = Movie("Palindrome Linked List", "Does the train read the same from both ends?")
    slow = fast = head
    m.step("Step 1: find the **middle** with 🐢 `slow` (1 step) and 🐇 `fast` (2 steps).",
           train(nodes, hi={slow: "look"}, ptr={"slow": slow, "fast": fast}))
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        m.step(f"🐢 on `{slow.val}`, 🐇 on `{fast.val if fast else '∅'}`." + (" 🐇 cannot jump again: `slow` is the middle." if not (fast and fast.next) else ""),
               train(nodes, hi={slow: "win", **({fast: "swap"} if fast else {})}, ptr={"slow": slow, "fast": fast}))
    prev = None
    flipped = []
    while slow:
        nxt = slow.next
        slow.next = prev
        flipped.append(slow)
        to = f"`{prev.val}`" if prev else "nothing (∅)"
        m.step(f"Step 2: reverse the back half. `{slow.val}` now holds {to} (orange arrows point left).",
               train(nodes, hi={**{f: "done" for f in flipped}, slow: "swap"}, ptr={"prev": slow, "slow": nxt}))
        prev, slow = slow, nxt
    left, right = head, prev
    same = []
    while right:
        if left.val != right.val:
            m.step(f"`{left.val}` ≠ `{right.val}`: the two ends differ → **False**.",
                   train(nodes, hi={left: "bad", right: "bad"}, ptr={"left": left, "right": right}))
            return False, m
        same += [left, right]
        m.step(f"Step 3: walk in from both ends. `{left.val}` = `{right.val}` ✔ so far so good.",
               train(nodes, hi={nd: "good" for nd in same}, ptr={"left": left, "right": right}))
        left, right = left.next, right.next
    m.step("The back-half finger ran out and every pair matched → **True** ✅",
           train(nodes, hi={nd: "good" for nd in nodes}))
    return True, m

values = [1, 2, 3, 2, 1]
answer, m = is_palindrome_movie(values)
assert answer == is_palindrome(build_list(values)) is True
m.show()

---

## 48 · Intersection of Two Linked Lists · [LeetCode 160](https://leetcode.com/problems/intersection-of-two-linked-lists/) · 🟢 Easy

**🧸 Like you're five:** Two trains join into one shared track, like a letter **Y**. Two friends start at the two tops. Each
        walks their own train and then the other one. They walk exactly the same distance, so they arrive at the
        joining carriage at the same moment.

**The question:** Given the heads of two singly linked lists, return the node where they **intersect** (the same node object, not just an equal value), or `None` if they never meet. No cycles. Do not change the lists. Follow-up: O(n + m) time, O(1) memory.

| Input | Output |
|---|---|
| `A = [4, 1, 8, 4, 5], B = [5, 6, 1, 8, 4, 5], shared from the 8` | `node 8` |
| `A = [1, 9, 1, 2, 4], B = [3, 2, 4], shared from the 2` | `node 2` |
| `A = [2, 6, 4], B = [1, 5], nothing shared` | `None` |

**🔎 How to spot it:** "two linked lists **merge / intersect** at some node" → the pointer-swap trick. The key difficulty is
        that the two lists have **different lengths**, so walking both side by side does not line up.

**💡 The trick (at the end, jump to the other train):** Pointer `a` walks list A; when it falls off the end it jumps to the head of B. Pointer `b` does the
        opposite. After `len(A) + len(B)` steps both have walked the same total distance, so they reach the shared node
        together — or both reach `None` together if there is none, which ends the loop too. Compare with `is`: equal
        values in different nodes (the `1`s above) are not an intersection.

**⏱ Cost:** **O(n + m)** time · **O(1)** space. A `set` of A's nodes, then walk B, is O(n + m) time but O(n) space; checking every pair is O(n·m).

In [ ]:
def get_intersection_node(headA, headB):
    a, b = headA, headB
    while a is not b:                          # same carriage (or both None) ends the walk
        a = a.next if a else headB             # at the end of A, jump to B
        b = b.next if b else headA             # at the end of B, jump to A
    return a

def make_y(a_only, b_only, shared):
    "Two trains that share their tail: returns (headA, headB, first shared node or None)."
    common = build_list(shared)
    def join(vals):
        head = build_list(vals)
        if head is None:
            return common
        last = head
        while last.next:
            last = last.next
        last.next = common
        return head
    return join(a_only), join(b_only), common

A, B, meet = make_y([4, 1], [5, 6, 1], [8, 4, 5])
assert get_intersection_node(A, B) is meet and meet.val == 8
A, B, meet = make_y([1, 9, 1], [3], [2, 4])
assert get_intersection_node(A, B) is meet
A, B, meet = make_y([2, 6, 4], [1, 5], [])
assert get_intersection_node(A, B) is None
A, B, meet = make_y([], [7, 7], [3])                     # A starts right at the shared node
assert get_intersection_node(A, B) is meet
rng = random.Random(0)
for _ in range(300):
    A, B, meet = make_y(*[[rng.randint(0, 3) for _ in range(rng.randint(0, 4))] for _ in range(3)])
    assert get_intersection_node(A, B) is meet
print(get_intersection_node(*make_y([4, 1], [5, 6, 1], [8, 4, 5])[:2]).val)

In [ ]:
def intersection_movie(a_only, b_only, shared):
    headA, headB, meet = make_y(a_only, b_only, shared)
    common = set(map(id, nodes_of(meet)))
    m = Movie("Intersection of Two Linked Lists", "Walk your train, then the other one; you meet where they join")
    def pic(a, b, win=None):
        def roles(head):
            hi = {nd: "win" for nd in nodes_of(head) if id(nd) in common}
            for f in (a, b):
                if f is not None:
                    hi[f] = "look"
            if win is not None:
                hi[win] = "good"
            return hi
        return (chain_of(headA, "train A", hi=roles(headA), ptr={"a": a, "b": b}),
                chain_of(headB, "train B", hi=roles(headB), ptr={"a": a, "b": b}))
    a, b, steps = headA, headB, 0
    m.step("Blue carriages are **shared**: both trains end in the very same carriages (drawn in both rows). `a` starts on A, `b` on B.",
           *pic(a, b), note(steps=0))
    while a is not b:
        a = a.next if a else headB
        b = b.next if b else headA
        steps += 1
        if a is b:
            break
        def where(f, name, other):
            return f"`{name}` fell off the end (∅); next step it jumps to train {other}" if f is None else f"`{name}` on `{f.val}`"
        m.step(where(a, "a", "B") + ", " + where(b, "b", "A") + ". Not the same carriage yet.", *pic(a, b), note(steps=steps))
    m.step(f"`a` and `b` stand on the **same** carriage `{a.val}`. Both walked the same distance ({steps} steps), so they arrived together ✅",
           *pic(a, b, win=a), note(steps=steps))
    return a, m

answer, m = intersection_movie([1, 9, 1], [3], [2, 4])
A, B, meet = make_y([1, 9, 1], [3], [2, 4])            # a fresh copy for the clean solution
assert get_intersection_node(A, B) is meet and answer.val == meet.val == 2
m.show()

---

## 49 · LRU Cache · [LeetCode 146](https://leetcode.com/problems/lru-cache/) · 🟡 Medium

**🧸 Like you're five:** A small toy shelf with room for only 2 toys. Every time you play with a toy, it moves to the **newest**
        end of the shelf. When a new toy arrives and the shelf is full, the toy at the **oldest** end — the one nobody
        touched for the longest time — goes back in the box.

**The question:** Design `LRUCache(capacity)` with `get(key)` (the value, or `-1` if missing) and `put(key, value)` (insert or update; if that makes it too full, evict the **least recently used** key). Both must run in **O(1)** average time. A `get` or `put` counts as a use.

| Input | Output |
|---|---|
| `LRUCache(2); put(1,1); put(2,2); get(1); put(3,3); get(2); put(4,4); get(1); get(3); get(4)` | `[1, -1, -1, 3, 4] from the gets` |
| `LRUCache(1); put(2,1); get(2); put(3,2); get(2); get(3)` | `[1, -1, 2]` |

**🔎 How to spot it:** "**least recently used**", "evict", "O(1) get and put" → you need two things at once: find a
        key fast (**hash map**) and keep keys in **order of use** with O(1) move-to-end and remove (**doubly linked
        list**). Python's `OrderedDict` is exactly both glued together.

**💡 The trick (hash map + doubly linked list):** Keep keys ordered from oldest to newest. `get`: if present, **move it to the newest end**, return the
        value. `put`: write the value, move the key to the newest end, and if the size is now over `capacity`, drop the
        oldest key. The common mistake is forgetting that `get` also counts as a use, or that `put` of an existing key
        must update *and* move it without evicting anything. In an interview, give the `OrderedDict` version, then show
        you can build it by hand: a dict `key → node` plus a doubly linked list between two guard nodes (`old` and
        `new`), so removing and appending a node is just re-hooking four hands.

**⏱ Cost:** **O(1)** average time per `get`/`put` · **O(capacity)** space. A plain list of keys would need O(n) to find and move a key.

In [ ]:
class LRUCache:
    "The interview answer: OrderedDict keeps keys from oldest (first) to newest (last)."
    def __init__(self, capacity):
        self.cap = capacity
        self.d = OrderedDict()

    def get(self, key):
        if key not in self.d:
            return -1
        self.d.move_to_end(key)             # used just now → newest end
        return self.d[key]

    def put(self, key, value):
        self.d[key] = value
        self.d.move_to_end(key)             # new or updated → newest end
        if len(self.d) > self.cap:
            self.d.popitem(last=False)      # too full → drop the oldest


class Node:
    def __init__(self, key=0, val=0):
        self.key, self.val, self.prev, self.next = key, val, None, None

class LRUCacheByHand:
    "The same cache built by hand: dict key → node, and a doubly linked list old ⇄ … ⇄ new."
    def __init__(self, capacity):
        self.cap, self.map = capacity, {}
        self.old, self.new = Node(), Node()          # two guard nodes, so no None checks
        self.old.next, self.new.prev = self.new, self.old

    def _remove(self, node):                          # unhook a node from its neighbours
        node.prev.next, node.next.prev = node.next, node.prev

    def _append(self, node):                          # hook a node in just before the `new` guard
        node.prev, node.next = self.new.prev, self.new
        self.new.prev.next = node
        self.new.prev = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node); self._append(node)       # move to the newest end
        return node.val

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = self.map[key] = Node(key, value)
        self._append(node)
        if len(self.map) > self.cap:
            lru = self.old.next                       # the oldest real node
            self._remove(lru)
            del self.map[lru.key]

def run(cls, capacity, ops):
    "Run ops like [('put', 1, 1), ('get', 1)] and return what the gets returned."
    c, out = cls(capacity), []
    for op in ops:
        if op[0] == "get":
            out.append(c.get(op[1]))
        else:
            c.put(op[1], op[2])
    return out

ops = [("put", 1, 1), ("put", 2, 2), ("get", 1), ("put", 3, 3), ("get", 2), ("put", 4, 4), ("get", 1), ("get", 3), ("get", 4)]
assert run(LRUCache, 2, ops) == run(LRUCacheByHand, 2, ops) == [1, -1, -1, 3, 4]
ops2 = [("put", 2, 1), ("get", 2), ("put", 3, 2), ("get", 2), ("get", 3)]
assert run(LRUCache, 1, ops2) == run(LRUCacheByHand, 1, ops2) == [1, -1, 2]
ops3 = [("put", 1, 1), ("put", 2, 2), ("put", 1, 10), ("put", 3, 3), ("get", 1), ("get", 2)]
assert run(LRUCache, 2, ops3) == [10, -1]            # updating 1 made it the newest, so 2 was evicted
rng = random.Random(0)
for _ in range(200):
    ops = [("get", rng.randint(1, 5)) if rng.random() < 0.5 else ("put", rng.randint(1, 5), rng.randint(0, 9))
           for _ in range(rng.randint(1, 25))]
    cap = rng.randint(1, 4)
    assert run(LRUCache, cap, ops) == run(LRUCacheByHand, cap, ops)
print(run(LRUCache, 2, [("put", 1, 1), ("put", 2, 2), ("get", 1), ("put", 3, 3), ("get", 2)]))

In [ ]:
def lru_movie(capacity, ops):
    m = Movie("LRU Cache", f"A shelf with room for {capacity}: the oldest-used toy leaves when it is full")
    d, out = OrderedDict(), []
    def pic(hi_q=None, hi_d=None):
        return (queue(list(d), "order of use: oldest → newest", hi=hi_q), hmap(dict(d), "the dict  key → value", hi=hi_d),
                note(capacity=capacity, gets=str(out)))
    m.step(f"An empty shelf with room for **{capacity}** keys. The line on the left shows who was used longest ago (front).", *pic())
    for op in ops:
        if op[0] == "get":
            key = op[1]
            if key not in d:
                out.append(-1)
                m.step(f"`get({key})`: `{key}` is not on the shelf → **-1**.", *pic())
                continue
            d.move_to_end(key)
            out.append(d[key])
            m.step(f"`get({key})` → **{d[key]}**. Using it makes `{key}` the **newest**, so it moves to the back of the line.",
                   *pic({len(d) - 1: "look"}, {key: "look"}))
        else:
            _, key, value = op
            d[key] = value
            d.move_to_end(key)
            if len(d) > capacity:
                oldest = next(iter(d))
                m.step(f"`put({key}, {value})`: now {len(d)} keys but room for {capacity}. The oldest-used key `{oldest}` must go.",
                       *pic({0: "bad", len(d) - 1: "new"}, {oldest: "bad", key: "new"}))
                d.popitem(last=False)
                m.step(f"`{oldest}` is evicted (thrown out). `{key}` sits at the newest end.", *pic({len(d) - 1: "new"}, {key: "new"}))
            else:
                m.step(f"`put({key}, {value})`: room left, so `{key}` joins at the **newest** end.", *pic({len(d) - 1: "new"}, {key: "new"}))
    m.step(f"All done. The gets returned **{out}** ✅", queue(list(d), "order of use: oldest → newest"),
           hmap(dict(d), "the dict  key → value"), note(gets=str(out)))
    return out, m

ops = [("put", 1, 1), ("put", 2, 2), ("get", 1), ("put", 3, 3), ("get", 2), ("put", 4, 4), ("get", 1), ("get", 3), ("get", 4)]
answer, m = lru_movie(2, ops)
assert answer == run(LRUCache, 2, ops) == run(LRUCacheByHand, 2, ops) == [1, -1, -1, 3, 4]
m.show()

---

## 50 · Maximum Depth of Binary Tree · [LeetCode 104](https://leetcode.com/problems/maximum-depth-of-binary-tree/) · 🟢 Easy

**🧸 Like you're five:** How many floors does this family tree have? Every person asks their two children *"how many floors
        are below you?"*, takes the bigger answer, and adds 1 for themselves. Nobody there? That's 0 floors.

**The question:** Return the maximum depth of a binary tree: the number of nodes on the longest path from the root down to a leaf. An empty tree has depth 0.

| Input | Output |
|---|---|
| `root = [3, 9, 20, None, None, 15, 7]` | `3` |
| `root = [1, None, 2]` | `2` |
| `root = []` | `0` |

**🔎 How to spot it:** "**depth / height** of a tree", or any question where a node's answer is built from its children's
        answers → recursion that returns a number. This is the "hello world" of tree recursion.

**💡 The trick (recursion: 1 + max(children)):** Base case: an empty node has depth `0`. Otherwise depth = `1 + max(depth(left), depth(right))`. Trust
        the recursion: you do not need to picture the whole tree, only one node and its two answers. (BFS counting
        levels also works — see problem 55.)

**⏱ Cost:** **O(n)** time (each node once) · **O(h)** space for the call stack, where h is the height (O(n) for a tree shaped like a line, O(log n) if balanced).

In [ ]:
def max_depth(root):
    if root is None:                    # empty spot: no floors
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

assert max_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 3
assert max_depth(build_tree([1, None, 2])) == 2
assert max_depth(build_tree([])) == 0
assert max_depth(build_tree([1])) == 1
assert max_depth(build_tree([1, 2, None, 3, None, 4])) == 4       # a tree shaped like a line
print(max_depth(build_tree([3, 9, 20, None, None, 15, 7])))

In [ ]:
def max_depth_movie(values):
    root = build_tree(values)
    m = Movie("Maximum Depth of Binary Tree", "Each node asks its children how deep they go, then adds 1")
    waiting, told, answered = [], {}, []
    def pic(extra=None):
        hi = {**{nd: "done" for nd in answered}, **{nd: "win" for nd in waiting}, **(extra or {})}
        return (tree(root, "the tree", hi=hi), stack([nd.val for nd in waiting], "waiting (call stack)"),
                hmap(told, "node → depth it reported"))
    m.step("Every node will answer *how many floors, counting me?* An empty spot answers `0`.", *pic())
    def depth(node):
        if node is None:
            return 0
        waiting.append(node)
        m.step(f"Enter `{node.val}`: before it can answer, it asks its left and right children (and waits).", *pic({node: "look"}))
        left, right = depth(node.left), depth(node.right)
        waiting.pop()
        told[node.val] = 1 + max(left, right)
        answered.append(node)
        m.step(f"`{node.val}` heard left = `{left}`, right = `{right}`. It tells its parent 1 + max({left}, {right}) = **{told[node.val]}**.",
               *pic({node: "new"}))
        return told[node.val]
    answer = depth(root)
    m.step(f"The root answers last: the tree is **{answer}** floors deep ✅", *pic({root: "good"}))
    return answer, m

values = [3, 9, 20, None, None, 15, 7]
answer, m = max_depth_movie(values)
assert answer == max_depth(build_tree(values)) == 3
m.show()

---

## 51 · Invert Binary Tree · [LeetCode 226](https://leetcode.com/problems/invert-binary-tree/) · 🟢 Easy

**🧸 Like you're five:** Hold the tree up to a mirror. Every parent swaps its left child and its right child, and then asks each
        child to do the same with their own children. Soon the whole tree is flipped.

**The question:** Invert a binary tree (mirror it left ↔ right) and return its root.

| Input | Output |
|---|---|
| `root = [4, 2, 7, 1, 3, 6, 9]` | `[4, 7, 2, 9, 6, 3, 1]` |
| `root = [2, 1, 3]` | `[2, 3, 1]` |
| `root = []` | `[]` |

**🔎 How to spot it:** "**invert / mirror / flip**" a tree → do one tiny thing at every node (swap its two children), and let
        recursion (or a queue) visit every node.

**💡 The trick (recursion: swap the children):** At each node: swap `node.left` and `node.right` in one line (`a, b = b, a`), then invert both
        children. Order does not matter (before or after the recursive calls), as long as you swap **every** node,
        not just the root's children. Returning the root at the end is part of the answer.

**⏱ Cost:** **O(n)** time · **O(h)** stack space. A BFS with a queue gives the same answer with O(width) space.

In [ ]:
def invert_tree(root):
    if root is None:
        return None
    root.left, root.right = root.right, root.left      # swap my two children
    invert_tree(root.left)                              # ... and let them swap theirs
    invert_tree(root.right)
    return root

assert to_level_list(invert_tree(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [4, 7, 2, 9, 6, 3, 1]
assert to_level_list(invert_tree(build_tree([2, 1, 3]))) == [2, 3, 1]
assert invert_tree(build_tree([])) is None
assert to_level_list(invert_tree(build_tree([1, 2]))) == [1, None, 2]    # a lone left child moves right
print(to_level_list(invert_tree(build_tree([4, 2, 7, 1, 3, 6, 9]))))

In [ ]:
def invert_tree_movie(values):
    root = build_tree(values)
    m = Movie("Invert Binary Tree", "Every node swaps its left and right child")
    done = []
    m.step("We want the mirror image. The plan: at every node, swap the two children, then let each child do the same.",
           tree(root, "the tree"))
    def invert(node):
        if node is None:
            return None
        node.left, node.right = node.right, node.left
        kids = [k for k in (node.left, node.right) if k]
        if kids:
            m.step(f"`{node.val}` swaps its children: now `{node.left.val if node.left else '∅'}` is on the left and `{node.right.val if node.right else '∅'}` on the right.",
                   tree(root, "the tree", hi={**{d: "done" for d in done}, node: "look", **{k: "swap" for k in kids}}))
        else:
            m.step(f"`{node.val}` is a leaf: no children to swap.", tree(root, "the tree", hi={**{d: "done" for d in done}, node: "look"}))
        done.append(node)
        invert(node.left)
        invert(node.right)
        return node
    invert(root)
    m.step(f"Every node swapped once: the tree is mirrored → **{to_level_list(root)}** ✅",
           tree(root, "the tree", hi={d: "good" for d in done}))
    return to_level_list(root), m

values = [4, 2, 7, 1, 3, 6, 9]
answer, m = invert_tree_movie(values)
assert answer == to_level_list(invert_tree(build_tree(values))) == [4, 7, 2, 9, 6, 3, 1]
m.show()

---

## 52 · Same Tree · [LeetCode 100](https://leetcode.com/problems/same-tree/) · 🟢 Easy

**🧸 Like you're five:** Two children built two block towers that branch like trees. Are they exactly the same? Put a finger on
        each top block: same colour? Then check the two left branches together and the two right branches together.

**The question:** Given the roots `p` and `q` of two binary trees, return `True` if they have the same shape and the same values at every node.

| Input | Output |
|---|---|
| `p = [1, 2, 3], q = [1, 2, 3]` | `True` |
| `p = [1, 2], q = [1, None, 2]` | `False` |
| `p = [1, 2, 1], q = [1, 1, 2]` | `False` |

**🔎 How to spot it:** "are two trees **the same / equal / mirror**?" → walk **two** trees with **one** recursion that takes
        a pair of nodes. (Used again in *Symmetric Tree* and *Subtree of Another Tree*.)

**💡 The trick (recursion on two trees at once):** Compare a pair of nodes: both empty → `True`; exactly one empty → `False` (different shape); different
        values → `False`; otherwise the left pair **and** the right pair must both be the same. Check the `None` cases
        *before* reading `.val`, or you crash. `and` stops at the first `False`, so a difference ends the walk early.

**⏱ Cost:** **O(min(n, m))** time — we stop at the first difference · **O(h)** stack space.

In [ ]:
def is_same_tree(p, q):
    if p is None and q is None:            # both empty: same
        return True
    if p is None or q is None:             # only one is empty: different shape
        return False
    return p.val == q.val and is_same_tree(p.left, q.left) and is_same_tree(p.right, q.right)

assert is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3])) is True
assert is_same_tree(build_tree([1, 2]), build_tree([1, None, 2])) is False
assert is_same_tree(build_tree([1, 2, 1]), build_tree([1, 1, 2])) is False
assert is_same_tree(build_tree([]), build_tree([])) is True
rng = random.Random(0)
for _ in range(300):
    a = [rng.choice([None, 1, 2]) for _ in range(rng.randint(0, 7))]
    b = [rng.choice([None, 1, 2]) for _ in range(rng.randint(0, 7))]
    assert is_same_tree(build_tree(a), build_tree(b)) == (to_level_list(build_tree(a)) == to_level_list(build_tree(b)))
print(is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3])))

In [ ]:
def is_same_tree_movie(pv, qv):
    p, q = build_tree(pv), build_tree(qv)
    m = Movie("Same Tree", "Walk both trees together, one pair of nodes at a time")
    ok, bad = [], []
    def pic(extra_p=None, extra_q=None):
        return (tree(p, "tree p", hi={**{a: "good" for a, _ in ok}, **{a: "bad" for a, _ in bad}, **(extra_p or {})}),
                tree(q, "tree q", hi={**{b: "good" for _, b in ok}, **{b: "bad" for _, b in bad}, **(extra_q or {})}))
    m.step("Put one finger on each root. At every pair we ask: same value? Then check left with left and right with right.", *pic())
    def same(a, b):
        if a is None and b is None:
            return True
        if a is None or b is None:
            bad.append((a, b))
            m.step("One side has a node and the other is empty → different shapes → **False**.", *pic({a: "bad"} if a else {}, {b: "bad"} if b else {}))
            return False
        if a.val != b.val:
            bad.append((a, b))
            m.step(f"`{a.val}` ≠ `{b.val}` → **False**, and `and` stops right here: nothing else is checked.", *pic({a: "bad"}, {b: "bad"}))
            return False
        m.step(f"`{a.val}` = `{b.val}` ✔ Now the two **left** children, then the two **right** children.", *pic({a: "look"}, {b: "look"}))
        ok.append((a, b))
        return same(a.left, b.left) and same(a.right, b.right)
    answer = same(p, q)
    m.step(f"Answer: **{answer}**" + (" ✅ every pair matched." if answer else " — the trees differ in one spot (red), and one spot is enough."),
           *pic())
    return answer, m

pv, qv = [1, 2, 3, 4, 5], [1, 2, 3, 4, 6]
answer, m = is_same_tree_movie(pv, qv)
assert answer == is_same_tree(build_tree(pv), build_tree(qv)) is False
m.show()

---

## 53 · Symmetric Tree · [LeetCode 101](https://leetcode.com/problems/symmetric-tree/) · 🟢 Easy

**🧸 Like you're five:** Fold a paper tree down the middle. Does the left half land exactly on the right half? The far-left
        leaf must meet the far-right leaf, and the two inner leaves must meet each other.

**The question:** Given the `root` of a binary tree, return `True` if it is a mirror of itself (symmetric around its centre).

| Input | Output |
|---|---|
| `root = [1, 2, 2, 3, 4, 4, 3]` | `True` |
| `root = [1, 2, 2, None, 3, None, 3]` | `False` |
| `root = [1]` | `True` |

**🔎 How to spot it:** "**symmetric / mirror** of itself" → it is *Same Tree* (problem 52) between the left and right
        subtree, but with the children **crossed**.

**💡 The trick (mirror pairs: outside with outside):** Write `mirror(a, b)`: both empty → `True`; one empty → `False`; values differ → `False`; else
        `mirror(a.left, b.right)` (the **outside** pair) and `mirror(a.right, b.left)` (the **inside** pair). The answer
        is `mirror(root.left, root.right)`. The usual mistake is comparing left with left, which checks "same", not
        "mirror".

**⏱ Cost:** **O(n)** time · **O(h)** stack space. (An iterative version puts pairs in a queue.)

In [ ]:
def is_symmetric(root):
    def mirror(a, b):
        if a is None and b is None:
            return True
        if a is None or b is None:
            return False
        return (a.val == b.val
                and mirror(a.left, b.right)          # outside pair
                and mirror(a.right, b.left))         # inside pair
    return root is None or mirror(root.left, root.right)

def mirror_copy(node):
    "A brute-force helper: build the mirror image as a new tree."
    return None if node is None else TreeNode(node.val, mirror_copy(node.right), mirror_copy(node.left))

assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])) is True
assert is_symmetric(build_tree([1, 2, 2, None, 3, None, 3])) is False
assert is_symmetric(build_tree([1])) is True
assert is_symmetric(build_tree([])) is True
rng = random.Random(0)
for _ in range(300):
    t = build_tree([rng.choice([None, 1, 2]) if i else 1 for i in range(rng.randint(1, 7))])
    assert is_symmetric(t) == (to_level_list(t) == to_level_list(mirror_copy(t)))
print(is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))

In [ ]:
def is_symmetric_movie(values):
    root = build_tree(values)
    m = Movie("Symmetric Tree", "Fold the tree down the middle: does each node land on its twin?")
    ok = []
    def pic(extra=None, ptr=None):
        return tree(root, "the tree", hi={**{nd: "good" for nd in ok}, **(extra or {})}, ptr=ptr)
    m.step("The root sits on the fold line, so it is its own twin. Now compare its **left** child with its **right** child.",
           pic({root: "look"}))
    ok.append(root)
    def mirror(a, b, kind):
        if a is None and b is None:
            return True
        if a is None or b is None or a.val != b.val:
            m.step(f"The {kind} pair does not match → **False**.", pic({x: "bad" for x in (a, b) if x}, {"a": a, "b": b}))
            return False
        m.step(f"The {kind} pair: `{a.val}` and `{b.val}` match ✔ Next: **outside** with outside, then **inside** with inside.",
               pic({a: "look", b: "look"}, {"a": a, "b": b}))
        ok.extend([a, b])
        return mirror(a.left, b.right, "outside") and mirror(a.right, b.left, "inside")
    answer = mirror(root.left, root.right, "first")
    m.step(f"Every twin pair matched → **{answer}** ✅" if answer else "Answer: **False**.", pic())
    return answer, m

values = [1, 2, 2, 3, 4, 4, 3, 5, None, None, 6, 6, None, None, 5]
answer, m = is_symmetric_movie(values)
assert answer == is_symmetric(build_tree(values)) is True
m.show()

---

## 54 · Diameter of Binary Tree · [LeetCode 543](https://leetcode.com/problems/diameter-of-binary-tree/) · 🟢 Easy

**🧸 Like you're five:** A spider wants the longest walk along the branches of a tree, from any leaf to any other leaf, never
        going back. At each branching point the longest walk that bends there is *the deepest reach on the left plus
        the deepest reach on the right*.

**The question:** Return the length (number of **edges**) of the longest path between any two nodes of a binary tree. The path does **not** have to pass through the root.

| Input | Output |
|---|---|
| `root = [1, 2, 3, 4, 5]` | `3   (4 → 2 → 1 → 3)` |
| `root = [1, 2]` | `1` |
| `root = [1]` | `0` |

**🔎 How to spot it:** "**longest path** between any two nodes" in a tree (diameter, max path sum, longest same-value path) →
        the recursion **returns one thing** (the height, one arm) but **records another** (the best path, two arms) in
        an outside variable.

**💡 The trick (return the height, remember the best):** `height(node)` returns the number of nodes on the longest way down (`None` → 0). At each node, the
        longest path that bends *here* has `height(left) + height(right)` edges — update `best` with it. Then return
        `1 + max(left, right)` to the parent, because a parent can only continue **one** arm. The classic mistake is
        returning `left + right` upward (a path cannot fork), or only checking the root.

**⏱ Cost:** **O(n)** time — one visit per node · **O(h)** stack space. Computing heights again at every node is O(n²).

In [ ]:
def diameter_of_binary_tree(root):
    best = 0
    def height(node):
        nonlocal best
        if node is None:
            return 0
        left, right = height(node.left), height(node.right)
        best = max(best, left + right)          # the longest path that bends at this node
        return 1 + max(left, right)             # my parent can only use one of my arms
    height(root)
    return best

assert diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])) == 3
assert diameter_of_binary_tree(build_tree([1, 2])) == 1
assert diameter_of_binary_tree(build_tree([1])) == 0
assert diameter_of_binary_tree(build_tree([1, 2, None, 3, 4, 5, None, None, 6])) == 4   # does not use the root
print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))

In [ ]:
def diameter_movie(values):
    root = build_tree(values)
    m = Movie("Diameter of Binary Tree", "Each node tells its parent its height, and checks the path that bends at itself")
    heights, done, best, best_at = {}, [], 0, None
    def pic(extra=None):
        return (tree(root, "the tree", hi={**{d: "done" for d in done}, **(extra or {})}),
                hmap(heights, "node → height it reported"), note(best=best))
    m.step("Each node returns its **height** (nodes on the longest way down). A path bending at a node is left height + right height edges.",
           *pic())
    def height(node):
        nonlocal best, best_at
        if node is None:
            return 0
        m.step(f"Enter `{node.val}`: ask the left child, then the right child, for their heights.", *pic({node: "look"}))
        left, right = height(node.left), height(node.right)
        bend = left + right
        if bend > best:
            best, best_at = bend, node
            news = f"a path bending here: {left} + {right} = **{bend}** edges, a new best!"
        else:
            news = f"a path bending here: {left} + {right} = {bend} edges, no better."
        heights[node.val] = 1 + max(left, right)
        done.append(node)
        m.step(f"`{node.val}`: {news} Tells its parent height 1 + max({left}, {right}) = `{heights[node.val]}`.",
               *pic({node: "new"}))
        return heights[node.val]
    height(root)
    m.step(f"Done. The longest path has **{best}** edges and bends at `{best_at.val}` — not at the root `{root.val}` ✅",
           *pic({best_at: "good"}))
    return best, m

values = [1, 2, None, 3, 4, 5, None, None, 6]
answer, m = diameter_movie(values)
assert answer == diameter_of_binary_tree(build_tree(values)) == 4
m.show()

---

## 55 · Binary Tree Level Order Traversal · [LeetCode 102](https://leetcode.com/problems/binary-tree-level-order-traversal/) · 🟡 Medium

**🧸 Like you're five:** Read a family tree floor by floor: grandma first, then her children from left to right, then all the
        grandchildren. Everyone waits in a line; when you read a person, their children join the back of the line.

**The question:** Return the level order traversal of a binary tree's node values: a list of levels, each level read left to right.

| Input | Output |
|---|---|
| `root = [3, 9, 20, None, None, 15, 7]` | `[[3], [9, 20], [15, 7]]` |
| `root = [1]` | `[[1]]` |
| `root = []` | `[]` |

**🔎 How to spot it:** "**level by level**", "row by row", "each depth", "closest to the root", "minimum depth" → BFS
        with a **queue** (`collections.deque`), not recursion.

**💡 The trick (bfs with a queue, one level at a time):** Put the root in a queue. While the queue is not empty: **the queue now holds exactly one whole level**,
        so remember `size = len(q)` and pop exactly that many nodes, writing their values into this level's list and
        pushing their children to the back. The `for _ in range(len(q))` line is the whole trick — without it you
        cannot tell where one level ends. Use `deque.popleft()` (O(1)), not `list.pop(0)` (O(n)).

**⏱ Cost:** **O(n)** time · **O(w)** space, where w is the widest level (up to about n/2).

In [ ]:
def level_order(root):
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):              # exactly the nodes of this level
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)          # children wait for the next level
            if node.right:
                q.append(node.right)
        out.append(level)
    return out

assert level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert level_order(build_tree([1])) == [[1]]
assert level_order(build_tree([])) == []
assert level_order(build_tree([1, 2, None, 3])) == [[1], [2], [3]]
print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))

In [ ]:
def level_order_movie(values):
    root = build_tree(values)
    m = Movie("Binary Tree Level Order Traversal", "Everyone waits in a line; reading a node sends its children to the back")
    out, q, done = [], deque([root]), []
    def pic(cur=None, level=None):
        hi = {**{d: "done" for d in done}, **{x: "win" for x in q}}
        if cur:
            hi[cur] = "look"
        answer = {i: lv for i, lv in enumerate(out)}
        if level is not None:
            answer[len(out)] = level
        return (tree(root, "the tree", hi=hi), queue([x.val for x in q], "queue (blue in the tree)"),
                hmap(answer, "answer  level → values"))
    m.step("Start: only the root is in the line (the queue).", *pic())
    while q:
        size = len(q)
        level = []
        m.step(f"Level {len(out)}: the queue holds exactly **{size}** node(s) = this whole level. We will pop exactly {size}.",
               *pic(level=level))
        for _ in range(size):
            node = q.popleft()
            level.append(node.val)
            kids = [k for k in (node.left, node.right) if k]
            q.extend(kids)
            joined = " and ".join(f"`{k.val}`" for k in kids)
            m.step(f"Pop `{node.val}` from the front and write it down. " + (f"Its children {joined} join the **back** of the line." if kids else "No children."),
                   *pic(node, level))
            done.append(node)
        out.append(level)
    m.step(f"The line is empty. Answer: **{out}** ✅", tree(root, "the tree", hi={d: "good" for d in done}),
           hmap({i: lv for i, lv in enumerate(out)}, "answer  level → values"))
    return out, m

values = [1, 2, 3, 4, 5, None, 6]
answer, m = level_order_movie(values)
assert answer == level_order(build_tree(values)) == [[1], [2, 3], [4, 5, 6]]
m.show()

---

## 56 · Binary Tree Right Side View · [LeetCode 199](https://leetcode.com/problems/binary-tree-right-side-view/) · 🟡 Medium

**🧸 Like you're five:** Stand on the right side of a tree-shaped building and look at it. On each floor you only see the
        person standing furthest right. Careful: on a floor where the right side is empty, you see someone from the
        left side!

**The question:** Given the `root` of a binary tree, imagine standing on its right side and return the values you can see, from top to bottom.

| Input | Output |
|---|---|
| `root = [1, 2, 3, None, 5, None, 4]` | `[1, 3, 4]` |
| `root = [1, 2, 3, 4]` | `[1, 3, 4]` |
| `root = []` | `[]` |

**🔎 How to spot it:** "**seen from the right / left side**", "last / first node of each level", "largest value in each
        row" → level-order BFS (problem 55) and keep one value per level.

**💡 The trick (bfs, keep the last of each level):** Do level-order BFS. In the `for i in range(size)` loop, the node with `i == size - 1` is the rightmost
        of its level: add it to the answer. The trap: "just keep going right" is **wrong** — a deeper node on the left
        (like `4` in `[1, 2, 3, 4]`) is visible when the right side is shorter. (DFS that visits right first and
        records the first node at each new depth also works.)

**⏱ Cost:** **O(n)** time · **O(w)** queue space.

In [ ]:
def right_side_view(root):
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            if i == size - 1:                # the last one on this floor is the one you see
                out.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    return out

assert right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert right_side_view(build_tree([1, None, 3])) == [1, 3]
assert right_side_view(build_tree([])) == []
assert right_side_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]      # 4 is on the LEFT side but still visible
rng = random.Random(0)
for _ in range(200):
    t = build_tree([rng.choice([None, 1, 2, 3]) if i else 0 for i in range(rng.randint(1, 12))])
    assert right_side_view(t) == [lv[-1] for lv in level_order(t)]
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))

In [ ]:
def right_side_view_movie(values):
    root = build_tree(values)
    m = Movie("Binary Tree Right Side View", "Read each floor left to right; the last one on a floor is the one you see")
    out, seen, done, q = [], [], [], deque([root])
    def pic(cur=None, role="look"):
        hi = {**{d: "done" for d in done}, **{x: "win" for x in q}, **{s: "good" for s in seen}}
        if cur:
            hi[cur] = role
        return (tree(root, "the tree", hi=hi), queue([x.val for x in q], "queue"), arr(out, "seen from the right"))
    m.step("Level-order BFS as in problem 55. On each floor, only the **last** node popped is visible from the right.", *pic())
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            q.extend(k for k in (node.left, node.right) if k)
            if i == size - 1:
                out.append(node.val)
                seen.append(node)
                m.step(f"`{node.val}` is the **last** of its floor ({size} node(s)), so we see it from the right 👀", *pic(node, "good"))
            else:
                m.step(f"`{node.val}` is not the last on its floor, so someone to its right hides it.", *pic(node))
            done.append(node)
    m.step(f"Seen from the right: **{out}**. Note `{out[-2]}` sits on the left half, but nothing on its floor is further right ✅",
           tree(root, "the tree", hi={s: "good" for s in seen}), arr(out, "seen from the right", hi={i: "good" for i in range(len(out))}))
    return out, m

values = [1, 2, 3, 4, 5, None, None, 7]
answer, m = right_side_view_movie(values)
assert answer == right_side_view(build_tree(values)) == [1, 3, 5, 7]
m.show()

---

## 57 · Validate Binary Search Tree · [LeetCode 98](https://leetcode.com/problems/validate-binary-search-tree/) · 🟡 Medium

**🧸 Like you're five:** In a sorting tree, everyone on your left must be smaller than you and everyone on your right bigger —
        not just your own children, but all of their children too. So when a child walks down, they carry a note:
        *"you must be bigger than this and smaller than that"*.

**The question:** Return `True` if a binary tree is a valid binary search tree: every node's left subtree holds only values **strictly less** than the node, its right subtree only values **strictly greater**, and both subtrees are BSTs too.

| Input | Output |
|---|---|
| `root = [2, 1, 3]` | `True` |
| `root = [5, 1, 4, None, None, 3, 6]` | `False` |
| `root = [5, 4, 6, None, None, 3, 7]` | `False   (3 is under 6, but also right of 5)` |

**🔎 How to spot it:** "**valid BST**", or any rule that comes from **all ancestors**, not just the parent → pass the
        allowed range down the recursion. (Or: an in-order walk of a BST is strictly increasing.)

**💡 The trick (pass the allowed range (low, high) down):** `ok(node, low, high)`: an empty node is fine; otherwise `low < node.val < high` must hold, then check
        `ok(node.left, low, node.val)` and `ok(node.right, node.val, high)`. Start with `(-inf, inf)`. The famous
        wrong answer only compares a node with its **own children** — it says `[5, 4, 6, None, None, 3, 7]` is valid,
        but `3` sits to the right of `5`. Equal values are not allowed (`<`, not `<=`).

**⏱ Cost:** **O(n)** time · **O(h)** stack space.

In [ ]:
def is_valid_bst(root):
    def ok(node, low, high):
        if node is None:
            return True
        if not (low < node.val < high):          # outside the range my ancestors allow
            return False
        return ok(node.left, low, node.val) and ok(node.right, node.val, high)
    return ok(root, float("-inf"), float("inf"))

def inorder(node):
    return inorder(node.left) + [node.val] + inorder(node.right) if node else []

assert is_valid_bst(build_tree([2, 1, 3])) is True
assert is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False
assert is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7])) is False    # the grandchild trap
assert is_valid_bst(build_tree([1, 1])) is False                         # duplicates are not allowed
assert is_valid_bst(build_tree([2147483647])) is True                    # a huge value is fine with inf
rng = random.Random(0)
for _ in range(300):
    t = build_tree([rng.choice([None, 1, 2, 3, 4, 5]) if i else 3 for i in range(rng.randint(1, 7))])
    vals = inorder(t)
    assert is_valid_bst(t) == all(a < b for a, b in zip(vals, vals[1:]))
print(is_valid_bst(build_tree([2, 1, 3])))

In [ ]:
def is_valid_bst_movie(values):
    root = build_tree(values)
    m = Movie("Validate Binary Search Tree", "Every node carries a note: the range its value must be inside")
    good, bad = [], []
    m.step("The root may be anything: range (-∞, ∞). Going **left**, the node's value becomes the new **high**; going **right**, the new **low**.",
           tree(root, "the tree"), note(low=float("-inf"), high=float("inf")))
    def ok(node, low, high):
        if node is None:
            return True
        if not (low < node.val < high):
            bad.append(node)
            m.step(f"`{node.val}` must be between `{low:g}` and `{high:g}`, but it is not! → **False**. (Its parent alone would not notice.)",
                   tree(root, "the tree", hi={**{g: "good" for g in good}, node: "bad"}, ptr={"here": node}),
                   note(low=low, value=node.val, high=high))
            return False
        good.append(node)
        m.step(f"`{low:g}` < `{node.val}` < `{high:g}` ✔ Its left side must stay below `{node.val}`, its right side above `{node.val}`.",
               tree(root, "the tree", hi={**{g: "good" for g in good}, node: "look"}, ptr={"here": node}),
               note(low=low, value=node.val, high=high))
        return ok(node.left, low, node.val) and ok(node.right, node.val, high)
    answer = ok(root, float("-inf"), float("inf"))
    why = (f"`{bad[0].val}` looked fine next to its own parent, but an ancestor higher up said no — only the carried range catches that."
           if bad else "Every node stayed inside its range ✅")
    m.step(f"Answer: **{answer}**. {why}", tree(root, "the tree", hi={**{g: "good" for g in good}, **{b: "bad" for b in bad}}))
    return answer, m

values = [5, 4, 6, None, None, 3, 7]
answer, m = is_valid_bst_movie(values)
assert answer == is_valid_bst(build_tree(values)) is False
m.show()

---

## 58 · Kth Smallest Element in a BST · [LeetCode 230](https://leetcode.com/problems/kth-smallest-element-in-a-bst/) · 🟡 Medium

**🧸 Like you're five:** In a sorting tree the smallest number hides at the far bottom-left. Walk *left, me, right* everywhere and
        you meet the numbers in order: 1st, 2nd, 3rd… Stop when you have counted to `k`.

**The question:** Given the `root` of a binary search tree and an integer `k` (1-indexed, `1 ≤ k ≤ n`), return the `k`-th smallest value in the tree.

| Input | Output |
|---|---|
| `root = [3, 1, 4, None, 2], k = 1` | `1` |
| `root = [5, 3, 6, 2, 4, None, None, 1], k = 3` | `3` |
| `root = [1], k = 1` | `1` |

**🔎 How to spot it:** "BST" + "**k-th smallest**" / "sorted order" / "closest value" → **in-order traversal** (left,
        node, right) visits a BST in increasing order. Do it with an explicit stack so you can **stop early**.

**💡 The trick (in-order is sorted; stop at k):** Keep a stack. **(1)** Go left as far as you can, pushing each node. **(2)** Pop: that is the next
        smallest; count it, and if the count reaches `k`, return it. **(3)** Move to the popped node's right child and
        repeat. Stopping at `k` makes it O(h + k) instead of visiting all n nodes. Watch the 1-indexing: `k = 1` is the
        smallest.

**⏱ Cost:** **O(h + k)** time · **O(h)** stack space. Collecting the whole in-order list first is O(n) time and space.

In [ ]:
def kth_smallest(root, k):
    stack, node = [], root
    while True:
        while node:                  # 1. slide down to the left, remembering the way back
            stack.append(node)
            node = node.left
        node = stack.pop()           # 2. the next smallest value
        k -= 1
        if k == 0:
            return node.val
        node = node.right            # 3. then everything bigger than it, on its right

assert kth_smallest(build_tree([3, 1, 4, None, 2]), 1) == 1
assert kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3
assert kth_smallest(build_tree([1]), 1) == 1
t = build_tree([5, 3, 6, 2, 4, None, None, 1])
assert [kth_smallest(t, k) for k in range(1, 7)] == [1, 2, 3, 4, 5, 6]
print(kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))

In [ ]:
def kth_smallest_movie(values, k):
    root = build_tree(values)
    m = Movie("Kth Smallest Element in a BST", f"Visit left, me, right: the values come out sorted. Stop at number {k}")
    todo, node, popped, count = [], root, [], 0             # todo is the stack of nodes
    def pic(extra=None):
        hi = {**{p: "done" for p in popped}, **{s: "win" for s in todo}, **(extra or {})}
        return (tree(root, "the BST", hi=hi), stack([s.val for s in todo], "stack (the way back up)"),
                arr([p.val for p in popped], "popped, in order"), note(count=count, k=k))
    m.step(f"In a BST, smaller values are on the left. Walking **left, me, right** meets them smallest-first. We want the {k}th.", *pic())
    while True:
        while node:
            todo.append(node)
            m.step(f"Slide left: push `{node.val}` on the stack (we come back to it later) and go to its left child.", *pic({node: "look"}))
            node = node.left
        node = todo.pop()
        count += 1
        popped.append(node)
        if count == k:
            m.step(f"No more left. Pop `{node.val}`: it is smallest number **{count}** = k → answer **{node.val}** ✅", *pic({node: "good"}))
            return node.val, m
        right = f"so go to its right child `{node.right.val}`" if node.right else "It has no right child, so pop again next"
        m.step(f"Nothing further left. Pop `{node.val}`: smallest number {count}. Not {k} yet{", " if node.right else ". "}{right}.", *pic({node: "look"}))
        node = node.right

values, k = [5, 3, 6, 2, 4, None, None, 1], 3
answer, m = kth_smallest_movie(values, k)
assert answer == kth_smallest(build_tree(values), k) == 3
m.show()

---

## 59 · Lowest Common Ancestor of a Binary Tree · [LeetCode 236](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/) · 🟡 Medium

**🧸 Like you're five:** Two cousins are lost in a big family tree. Every person asks their two children *"did you find one of
        the cousins below you?"* The first person who hears *yes* from **both** sides is the closest grandparent they
        share.

**The question:** Given a binary tree and two different nodes `p` and `q` that are both in it, return their lowest common ancestor: the deepest node that has both `p` and `q` below it (a node counts as below itself).

| Input | Output |
|---|---|
| `root = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p = 5, q = 1` | `3` |
| `root = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p = 5, q = 4` | `5   (a node can be its own ancestor)` |
| `root = [1, 2], p = 1, q = 2` | `1` |

**🔎 How to spot it:** "**lowest common ancestor**", "closest shared parent", "where do two paths meet" in a tree →
        post-order recursion that reports *what I found* upward. (For a **BST** there is an easier walk: go left if both
        are smaller, right if both are bigger.)

**💡 The trick (ask both sides: did you find p or q?):** `lca(node)`: if `node` is empty, or is `p` or `q`, return `node`. Otherwise ask `left = lca(node.left)`
        and `right = lca(node.right)`. If **both** are not empty, `p` and `q` are on different sides — `node` is the
        answer. Otherwise pass up whichever side found something (`left or right`). Returning early at `p` is safe:
        if `q` is under `p`, then `p` is the answer anyway.

**⏱ Cost:** **O(n)** time · **O(h)** stack space. Brute force: store the root-to-node path for p and q, then find where they split — also O(n), but more code and O(n) space.

In [ ]:
def lowest_common_ancestor(root, p, q):
    if root is None or root is p or root is q:
        return root                                   # empty, or found one of them
    left = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)
    if left and right:                                # one on each side: I am the meeting point
        return root
    return left or right                              # pass up whatever I found (or None)

def find(root, val):
    "The node holding `val` (values are unique in these trees)."
    if root is None or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert lowest_common_ancestor(t, find(t, 5), find(t, 1)).val == 3
assert lowest_common_ancestor(t, find(t, 5), find(t, 4)).val == 5
assert lowest_common_ancestor(t, find(t, 7), find(t, 8)).val == 3
assert lowest_common_ancestor(t, find(t, 6), find(t, 4)).val == 5
t2 = build_tree([1, 2])
assert lowest_common_ancestor(t2, find(t2, 1), find(t2, 2)).val == 1
print(lowest_common_ancestor(t, find(t, 5), find(t, 1)).val)

In [ ]:
def lca_movie(values, pv, qv):
    root = build_tree(values)
    p, q = find(root, pv), find(root, qv)
    m = Movie("Lowest Common Ancestor of a Binary Tree", f"Where do the paths to {pv} and {qv} meet?")
    said, done = {}, []
    def pic(extra=None):
        hi = {**{d: "done" for d in done}, p: "new", q: "new", **(extra or {})}
        return tree(root, "the tree (purple = p and q)", hi=hi, ptr={"p": p, "q": q}), hmap(said, "node → what it told its parent")
    m.step(f"Every node asks both children *did you find `{pv}` or `{qv}`?* and passes the news up. ∅ means *found nothing*.", *pic())
    def lca(node):
        if node is None:
            return None
        if node is p or node is q:
            said[node.val] = node.val
            done.append(node)
            m.step(f"`{node.val}` **is** one of the two! It returns itself at once, without looking below.", *pic({node: "look"}))
            return node
        leaf = node.left is None and node.right is None
        if not leaf:
            m.step(f"Enter `{node.val}`: ask the left child, then the right child.", *pic({node: "look"}))
        left, right = lca(node.left), lca(node.right)
        done.append(node)
        if left and right:
            said[node.val] = node.val
            m.step(f"`{node.val}` heard **yes** from both sides (`{left.val}` and `{right.val}`): the two are split here, so `{node.val}` is the answer, and it passes itself up.",
                   *pic({node: "good"}))
            return node
        found = left or right
        said[node.val] = found.val if found else "∅"
        if leaf:
            m.step(f"`{node.val}` is a leaf and not `{pv}` or `{qv}`: it tells its parent ∅.", *pic({node: "look"}))
        elif found:
            m.step(f"`{node.val}` heard `{found.val}` from one side only, so it just passes `{found.val}` up.", *pic({node: "look"}))
        else:
            m.step(f"`{node.val}` found nothing on either side: it tells its parent ∅.", *pic({node: "look"}))
        return found
    answer = lca(root)
    m.step(f"The root passes up `{answer.val}`: the lowest common ancestor is **{answer.val}** ✅", *pic({answer: "good"}))
    return answer.val, m

values = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
answer, m = lca_movie(values, 6, 4)
t = build_tree(values)
assert answer == lowest_common_ancestor(t, find(t, 6), find(t, 4)).val == 5
m.show()

---

## 60 · Construct Binary Tree from Preorder and Inorder Traversal · [LeetCode 105](https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/) · 🟡 Medium

**🧸 Like you're five:** Two lists describe one tree. The first list always says the **boss** first. The second list has
        everyone on the boss's left team before the boss and the right team after. Find the boss in the second list and
        you know who is on which team — then do the same inside each team.

**The question:** Given `preorder` and `inorder` traversals of a binary tree with **unique** values, build and return the tree.

| Input | Output |
|---|---|
| `preorder = [3, 9, 20, 15, 7], inorder = [9, 3, 15, 20, 7]` | `[3, 9, 20, None, None, 15, 7]` |
| `preorder = [-1], inorder = [-1]` | `[-1]` |
| `preorder = [1, 2], inorder = [2, 1]` | `[1, 2]` |

**🔎 How to spot it:** "**construct / rebuild** a tree from two traversals" → the preorder (or postorder) list tells you the
        **root**, the inorder list tells you how many nodes go **left** and **right**. Unique values make the split
        exact.

**💡 The trick (preorder names the root, inorder splits left and right):** Build a dict `value → index in inorder` once, so finding the root is O(1). Walk preorder with one
        shared index. `make(lo, hi)` builds the subtree whose values are `inorder[lo..hi]`: take the next preorder value
        as the root, find it at `mid` in inorder, build the **left** from `lo..mid-1` first, then the **right** from
        `mid+1..hi`. Left before right matters, because preorder lists the whole left team before the right team.
        Slicing lists (`inorder[:mid]`) works too, but costs O(n²).

**⏱ Cost:** **O(n)** time with the index dict · **O(n)** space (dict + tree; recursion O(h)). The slicing version is O(n²).

In [ ]:
def build_tree_pre_in(preorder, inorder):
    # LeetCode calls this buildTree; our helper build_tree(level_order) already has that name.
    where = {v: i for i, v in enumerate(inorder)}      # value -> its place in inorder
    nxt = iter(preorder)                                 # preorder names the roots, in order
    def make(lo, hi):                                    # build from inorder[lo..hi]
        if lo > hi:
            return None
        root = TreeNode(next(nxt))
        mid = where[root.val]
        root.left = make(lo, mid - 1)                    # left team first (preorder lists it first)
        root.right = make(mid + 1, hi)
        return root
    return make(0, len(inorder) - 1)

def preorder_of(node):
    return [node.val] + preorder_of(node.left) + preorder_of(node.right) if node else []

def inorder_of(node):
    return inorder_of(node.left) + [node.val] + inorder_of(node.right) if node else []

assert to_level_list(build_tree_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])) == [3, 9, 20, None, None, 15, 7]
assert to_level_list(build_tree_pre_in([-1], [-1])) == [-1]
assert to_level_list(build_tree_pre_in([1, 2], [2, 1])) == [1, 2]
assert to_level_list(build_tree_pre_in([1, 2], [1, 2])) == [1, None, 2]
rng = random.Random(0)
for _ in range(200):
    shape = [1 if i == 0 else rng.choice([None, 1, 1]) for i in range(rng.randint(1, 12))]
    t = build_tree(shape)
    labels = iter(rng.sample(range(-50, 50), 12))
    q = deque([t])                                       # give every node a different value
    while q:
        nd = q.popleft(); nd.val = next(labels)
        q.extend(k for k in (nd.left, nd.right) if k)
    assert to_level_list(build_tree_pre_in(preorder_of(t), inorder_of(t))) == to_level_list(t)
print(to_level_list(build_tree_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))

In [ ]:
def build_tree_movie(preorder, inorder):
    m = Movie("Construct Binary Tree from Preorder and Inorder", "Preorder says who the boss is; inorder says who is on the left and right team")
    where = {v: i for i, v in enumerate(inorder)}
    i, top, made = 0, [None], []
    m.step("Preorder lists every boss **before** its teams. Inorder lists the left team, then the boss, then the right team.",
           arr(preorder, "preorder", ptr={"next": 0}), arr(inorder, "inorder"), tree(None, "the tree so far"))
    def make(lo, hi, attach, side):
        nonlocal i
        if lo > hi:
            return None
        node = TreeNode(preorder[i])
        attach(node)
        mid = where[node.val]
        made.append(node)
        left_team, right_team = inorder[lo:mid], inorder[mid + 1:hi + 1]
        m.step(f"Next in preorder: `{node.val}`, boss of {side}. Inorder puts {left_team or 'nobody'} on its left, {right_team or 'nobody'} on its right.",
               arr(preorder, "preorder", hi={**{j: "done" for j in range(i)}, i: "look"}, ptr={"next": i}),
               arr(inorder, "inorder", hi={mid: "good"}, win=(lo, hi)),
               tree(top[0], "the tree so far", hi={**{x: "done" for x in made}, node: "new"}))
        i += 1
        make(lo, mid - 1, lambda c: setattr(node, "left", c), f"the left team of `{node.val}`")
        make(mid + 1, hi, lambda c: setattr(node, "right", c), f"the right team of `{node.val}`")
        return node
    make(0, len(inorder) - 1, lambda c: top.__setitem__(0, c), "the whole tree")
    answer = to_level_list(top[0])
    m.step(f"Every preorder value has been used, left team always before right team. The tree: **{answer}** ✅",
           arr(preorder, "preorder", hi={j: "done" for j in range(len(preorder))}), arr(inorder, "inorder"),
           tree(top[0], "the tree", hi={x: "good" for x in made}))
    return answer, m

preorder, inorder_list = [3, 9, 1, 20, 15, 7], [1, 9, 3, 15, 20, 7]
answer, m = build_tree_movie(preorder, inorder_list)
assert answer == to_level_list(build_tree_pre_in(preorder, inorder_list)) == [3, 9, 20, 1, None, 15, 7]
m.show()

---

## Quiz yourself: which trick?

Eight more LeetCode questions that are *not* in the top 100. Read each one, name the trick out loud, then open the
answer.

1. **[Remove Duplicates from Sorted List](https://leetcode.com/problems/remove-duplicates-from-sorted-list/)** (83, Easy) — delete repeated values from a sorted linked list.
   <details><summary>which trick?</summary>Walk and re-hook (pattern 1). Sorted means repeats sit next to each other, so while <code>cur.next.val == cur.val</code>, skip it with <code>cur.next = cur.next.next</code>.</details>
2. **[Swap Nodes in Pairs](https://leetcode.com/problems/swap-nodes-in-pairs/)** (24, Medium) — swap every two adjacent nodes, without changing values.
   <details><summary>which trick?</summary>Dummy head + re-hooking hands (problems 41, 45). The head changes, so a dummy saves a special case; each swap is three hand changes.</details>
3. **[Odd Even Linked List](https://leetcode.com/problems/odd-even-linked-list/)** (328, Medium) — all odd-position nodes first, then the even ones, in O(1) space.
   <details><summary>which trick?</summary>Two small trains grown while walking once (like problem 42 in reverse), then hook the end of the odd train to the head of the even train.</details>
4. **[Rotate List](https://leetcode.com/problems/rotate-list/)** (61, Medium) — rotate a linked list to the right by <code>k</code> places.
   <details><summary>which trick?</summary>Count the length, take <code>k % length</code>, then the new tail is <code>length - k</code> steps in — the "n-th from the end" idea of problem 45 (or join the tail to the head and cut the ring).</details>
5. **[Path Sum](https://leetcode.com/problems/path-sum/)** (112, Easy) — is there a root-to-leaf path whose values add up to <code>target</code>?
   <details><summary>which trick?</summary>Recursion that passes something <b>down</b> (the amount still needed), like the range in problem 57; the check happens only at a leaf.</details>
6. **[Average of Levels in Binary Tree](https://leetcode.com/problems/average-of-levels-in-binary-tree/)** (637, Easy) — the average value on each level.
   <details><summary>which trick?</summary>"Each level" → BFS with <code>for _ in range(len(q))</code>, exactly problem 55; just sum and divide.</details>
7. **[Minimum Absolute Difference in BST](https://leetcode.com/problems/minimum-absolute-difference-in-bst/)** (530, Easy) — the smallest difference between any two values in a BST.
   <details><summary>which trick?</summary>In-order walk of a BST is sorted (problem 58), and in a sorted list the closest pair is always two neighbours.</details>
8. **[Count Good Nodes in Binary Tree](https://leetcode.com/problems/count-good-nodes-in-binary-tree/)** (1448, Medium) — count nodes with no bigger value on the path from the root.
   <details><summary>which trick?</summary>A rule that depends on <b>all ancestors</b> → pass the biggest value so far down the recursion, like <code>(low, high)</code> in problem 57.</details>

## Pattern cheat sheet

| If the question says… | think… | example here |
|---|---|---|
| "reverse", or needs part of a list backwards | three fingers `prev`, `cur`, `nxt`; save `nxt` before flipping | 41, 47 |
| build a new list / the head might change or be removed | a **dummy** node in front, return `dummy.next` | 42, 45, 46 |
| "cycle", "middle", "O(1) memory" on a list | **fast and slow** fingers | 43, 44, 47 |
| "n-th from the end" | two fingers with a gap of `n` | 45 |
| two lists of different lengths that meet | walk yours, then the other one | 48 |
| "O(1) get and put", "least recently used" | dict + doubly linked list (`OrderedDict`) | 49 |
| "depth", "height", "count", a node's answer comes from its children | recursion: ask the children, tell the parent | 50, 51 |
| "same", "mirror", "symmetric" | one recursion on a **pair** of nodes | 52, 53 |
| "longest path between any two nodes" | return the height, record `left + right` in a `best` | 54 |
| "level", "row", "seen from the side", "nearest to the root" | BFS with a queue, `for _ in range(len(q))` | 55, 56 |
| a rule from **all** ancestors ("valid BST") | pass a range `(low, high)` down | 57 |
| BST + "k-th smallest" / "sorted order" | in-order walk (left, me, right) with a stack, stop early | 58 |
| "lowest common ancestor" | return what you found; found on both sides → it is you | 59 |
| "construct from preorder and inorder" | preorder gives the root, an index dict on inorder splits | 60 |

## Say it in an interview

- **Clarify first:** can the list or tree be empty? Can values repeat (this breaks "compare by value" in 48 and the
  BST rules in 57)? May I change the input list (47)? Is it a plain binary tree or a **BST** (59 is much easier
  on a BST)?
- **Brute force in one sentence, then improve:** "I could copy the list into a Python list — O(n) space; with fast
  and slow pointers I can do it in O(1)." Interviewers want to hear you *see* the trade-off.
- **Name the trick out loud:** "this is fast and slow pointers", "I'll add a dummy head so removing the first node
  is not a special case", "this is post-order recursion: each node combines its children's answers".
- **Say the cost with the right letters:** trees cost **O(n)** time and **O(h)** space for recursion — h is the
  height, which is O(log n) when balanced but O(n) for a tree shaped like a line (Python's default recursion limit
  is about 1000, so mention an iterative version for very deep trees).
- **Test the edge cases aloud:** empty (`None`), one node, two nodes, the answer at the head or root, an even vs
  odd length list, a lopsided tree.
- **Common traps:** flipping `cur.next` before saving it (41); `fast.next.next` without checking `fast.next` (43);
  comparing nodes by value instead of `is` (43, 48); forgetting the final carry (46); checking only a node's own
  children for a BST (57); returning `left + right` upward instead of one arm (54).

## Next

- Previous: [16 · Top 100, part 2 — sliding window, stacks and binary search](16_top100_part2_sliding_window_stack_binary_search.ipynb)
- Next: [18 · Top 100, part 4 — heaps, backtracking and graphs](18_top100_part4_heaps_backtracking_graphs.ipynb)
- Go deeper: [05 · Linked lists](05_linked_lists.ipynb) · [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb) · [09 · Trees and BST](09_trees_and_bst.ipynb) · [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb)